## Audio RAG — Whisper (Full-Text) vs CLAP (Shared Semantic Space)

Experiments on audio input (MP3 narration of the Transformer paper, *Attention Is All You Need*).

### Pipeline comparison

| Aspect | Whisper pipeline | CLAP pipeline |
|---|---|---|
| Indexing | Audio → Whisper transcription → text embedding | Audio segments → CLAP audio encoder → audio embedding |
| Query encoding | Text embedding model | CLAP **text** encoder (same shared space) |
| Retrieval | Text ↔ Text similarity | Text ↔ Audio cross-modal similarity |
| Context for LLM | Transcribed text chunks | Whisper transcription of **retrieved** audio segments |
| BM25 | On transcription | Not applicable |
| Qdrant collection | `QDRANT_WHISPER_COLLECTION` | `QDRANT_CLAP_COLLECTION` |

> **Key insight**: in both pipelines the LLM receives text. What differs is *how* the relevant
> segments are found: keyword/semantic similarity on transcription (Whisper) vs
> cross-modal similarity in the CLIP-style audio-text space (CLAP).


### Evaluation
The final section compares Whisper and CLAP RAG with BERTScore, answer precision/recall, context recall, timestamp coverage, and must/should claim recall. Evaluation generation is cached separately from the interactive filter table, and each approach exposes only retrieval modes that make sense for that pipeline.


### 1. Configuration


In [24]:
import hashlib, os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
for _env_path in ("../setup.env", "setup.env"):
    if os.path.exists(_env_path):
        load_dotenv(_env_path, override=True)

# Reuse the Hugging Face cache downloaded on Windows when this notebook runs
# inside WSL. The WSL virtualenv also installs this behavior at Python startup
# via install_wsl_hf_cache_hook.py, but this fallback helps with other kernels.
def _truthy_env(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() not in {"0", "false", "no", "off"}


def _running_in_wsl() -> bool:
    if os.getenv("WSL_DISTRO_NAME"):
        return True
    try:
        import platform
        return "microsoft" in platform.release().lower()
    except Exception:
        return False


def _configure_wsl_hf_cache() -> None:
    if not _running_in_wsl():
        return
    hf_home = os.getenv("WSL_WINDOWS_HF_HOME", "/mnt/c/Users/danie/.cache/huggingface")
    if not os.path.isdir(hf_home):
        print(f"  [warn] WSL Windows Hugging Face cache not found: {hf_home}")
        return
    force_cache = _truthy_env("MULTIRAG_FORCE_WSL_HF_CACHE", True)

    def set_cache_var(name: str, value: str) -> None:
        if force_cache or not os.getenv(name):
            os.environ[name] = value

    set_cache_var("HF_HOME", hf_home)
    set_cache_var("HF_HUB_CACHE", os.path.join(hf_home, "hub"))
    set_cache_var("HF_DATASETS_CACHE", os.path.join(hf_home, "datasets"))

    if _truthy_env("MULTIRAG_HF_OFFLINE", True) and not _truthy_env("MULTIRAG_HF_ALLOW_DOWNLOADS", False):
        os.environ.setdefault("HF_HUB_OFFLINE", "1")
        os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

    print(
        f"  Hugging Face cache: {os.environ['HF_HOME']} "
        f"(Windows cache mounted in WSL, offline={os.environ.get('HF_HUB_OFFLINE') == '1'})"
    )


_configure_wsl_hf_cache()

# ── Audio input ───────────────────────────────────────────────────────────────
# Path to the MP3 file. Accepted formats: mp3, wav, m4a, flac.
# Tip: download from YouTube with:
#   yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <URL>
AUDIO_PATH = os.getenv("AUDIO_PATH", "./content/audio.mp3")


def _audio_source_id(path: str, length: int = 12) -> str:
    """Return a stable fingerprint so indexes from different audio files never mix."""
    if not os.path.exists(path):
        return "missing"
    digest = hashlib.md5()
    with open(path, "rb") as audio_file:
        for chunk in iter(lambda: audio_file.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()[:length]


AUDIO_SOURCE_ID = _audio_source_id(AUDIO_PATH)

# ── Whisper (Speech-to-Text) ──────────────────────────────────────────────────
#
# Backends:
#   "faster-whisper"  — recommended, fast GPU inference via CTranslate2
#   "hf"              — fallback using transformers.pipeline
#
# Good whisper model choices:
#   "deepdml/faster-whisper-large-v3-turbo-ct2" — fast + high quality
#   "Systran/faster-whisper-large-v3"           — better quality, slower
#   "Systran/faster-whisper-medium"             — smaller/faster fallback
WHISPER_BACKEND = os.getenv("WHISPER_BACKEND", "faster-whisper")
# Run native CTranslate2 work outside Jupyter so a CUDA-level failure cannot
# terminate the notebook kernel. The child process uses the kernel Python.
WHISPER_SUBPROCESS = os.getenv("WHISPER_SUBPROCESS", "true").lower() == "true"
WHISPER_SUBPROCESS_CPU_FALLBACK = os.getenv("WHISPER_SUBPROCESS_CPU_FALLBACK", "true").lower() == "true"
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "deepdml/faster-whisper-large-v3-turbo-ct2")
WHISPER_LANGUAGE = os.getenv("WHISPER_LANGUAGE", "en")   # set to empty/None for auto-detect
# Sequential CTranslate2 inference is the stable default for Jupyter kernels.
# BatchedInferencePipeline can terminate the native kernel process on some
# Windows/CUDA combinations. Set WHISPER_INFERENCE_MODE=batched explicitly
# after the sequential path works, and raise the batch size gradually.
WHISPER_INFERENCE_MODE = os.getenv("WHISPER_INFERENCE_MODE", "sequential").lower()
WHISPER_BATCH_SIZE = int(os.getenv("WHISPER_BATCH_SIZE", "4"))  # only used by batched/HF inference
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "cuda")         # "cuda" | "cpu" | "auto"
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "float16") # use "int8_float16" if low VRAM
WHISPER_BEAM_SIZE = int(os.getenv("WHISPER_BEAM_SIZE", "1"))     # 1 = fastest; 5 = potentially better quality
WHISPER_VAD_FILTER = os.getenv("WHISPER_VAD_FILTER", "true").lower() == "true"

# ── Audio chunking ────────────────────────────────────────────────────────────
# Whisper returns segment-level timestamps. We merge consecutive segments
# into chunks of at most AUDIO_CHUNK_SECS seconds for indexing.
AUDIO_CHUNK_SECS  = int(os.getenv("AUDIO_CHUNK_SECS",   "45"))   # max seconds per text chunk
AUDIO_CHUNK_OVERLAP_SECS = int(os.getenv("AUDIO_CHUNK_OVERLAP_SECS", "5"))  # overlap between chunks

# CLAP splits the raw audio into fixed-length segments for embedding.
CLAP_SEGMENT_SECS = int(os.getenv("CLAP_SEGMENT_SECS",    "10"))  # seconds per CLAP segment
CLAP_SEGMENT_OVERLAP = int(os.getenv("CLAP_SEGMENT_OVERLAP",  "2"))  # overlap in seconds

# ── CLAP model ────────────────────────────────────────────────────────────────
# "laion/larger_clap_general"   — best general-purpose (music + speech + AudioSet)
# "laion/larger_clap_music"     — music-specialised
# "laion/clap-htsat-unfused"    — lighter baseline
CLAP_MODEL = os.getenv("CLAP_MODEL", "laion/larger_clap_general")
# CLAP_MODEL      = os.getenv("CLAP_MODEL", "laion/clap-htsat-unfused")

# ── Text embedding (Whisper pipeline) ─────────────────────────────────────────
# Same model as rag_pipeline_local.ipynb for fair comparison.
# "BAAI/bge-m3"                              — recommended (1024-dim)
# "sentence-transformers/all-MiniLM-L6-v2"  — lightweight (384-dim)
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVER_K = int(os.getenv("RETRIEVER_K",    "8"))
RERANKER_TOP_N = int(os.getenv("RERANKER_TOP_N", "4"))
RERANKER_MODEL = os.getenv("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
ENABLE_BM25 = os.getenv("ENABLE_BM25", "true").lower() == "true"
ENABLE_RERANKING  = os.getenv("ENABLE_RERANKING", "true").lower() == "true"

# ── Qdrant (local Docker) ─────────────────────────────────────────────────────
# docker run -d --name qdrant -p 6333:6333 -p 6334:6334 \
#   -v $(pwd)/qdrant_storage:/qdrant/storage qdrant/qdrant
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
QDRANT_WHISPER_COLLECTION = os.getenv("QDRANT_WHISPER_COLLECTION", f"audio_whisper_{AUDIO_SOURCE_ID}")
QDRANT_CLAP_COLLECTION = os.getenv("QDRANT_CLAP_COLLECTION", f"audio_clap_{AUDIO_SOURCE_ID}")
RESET_WHISPER_COLLECTION = os.getenv("RESET_WHISPER_COLLECTION", "false").lower() == "true"
RESET_CLAP_COLLECTION = os.getenv("RESET_CLAP_COLLECTION", "false").lower() == "true"

# ── Generation model ──────────────────────────────────────────────────────────
# Text-only generation (audio pipelines never pass raw audio to the LLM).
# "Qwen/Qwen2.5-3B-Instruct"          — local, CPU/GPU
# "Qwen/Qwen2.5-VL-3B-Instruct"       — also works (text-only prompt)
# Or use Ollama: set GENERATION_BACKEND=ollama and GENERATION_MODEL=mistral-nemo
GENERATION_MODEL = os.getenv("GENERATION_MODEL",   "Qwen/Qwen2.5-3B-Instruct")
GENERATION_BACKEND  = os.getenv("GENERATION_BACKEND", "hf")  # "hf" | "ollama"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "")
GENERATION_MAX_NEW_TOKENS = int(os.getenv("GENERATION_MAX_NEW_TOKENS", "512"))
HF_DEVICE_MAP = os.getenv("HF_DEVICE_MAP",  "auto")
HF_TORCH_DTYPE = os.getenv("HF_TORCH_DTYPE", "auto")

# ── Persistence ───────────────────────────────────────────────────────────────
PERSIST_DIR = os.getenv("PERSIST_DIR", "./cache/audio/")
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs("./content/", exist_ok=True)

print("Configuration loaded.")
print(f"  Audio      : {AUDIO_PATH} (source={AUDIO_SOURCE_ID})")
print(f"  Whisper    : {WHISPER_MODEL} | backend={WHISPER_BACKEND} | lang={WHISPER_LANGUAGE or 'auto'}")
print(
    f"  Whisper HW : device={WHISPER_DEVICE}, compute={WHISPER_COMPUTE_TYPE}, "
    f"isolated={WHISPER_SUBPROCESS}, mode={WHISPER_INFERENCE_MODE}, "
    f"batch={WHISPER_BATCH_SIZE}, beam={WHISPER_BEAM_SIZE}"
)
print(f"  Chunk      : {AUDIO_CHUNK_SECS}s (overlap={AUDIO_CHUNK_OVERLAP_SECS}s)")
print(f"  CLAP       : {CLAP_MODEL} | segment={CLAP_SEGMENT_SECS}s (overlap={CLAP_SEGMENT_OVERLAP}s)")
print(f"  Embeddings : {EMBEDDING_MODEL}")
print(f"  Retrieval  : k={RETRIEVER_K}, reranker_top_n={RERANKER_TOP_N}")
print(f"  Generator  : {GENERATION_MODEL} ({GENERATION_BACKEND})")
if OLLAMA_BASE_URL:
    print(f"  Ollama URL : {OLLAMA_BASE_URL}")
print(f"  Qdrant     : {QDRANT_URL}")
print(f"    Whisper collection : {QDRANT_WHISPER_COLLECTION}")
print(f"    CLAP collection    : {QDRANT_CLAP_COLLECTION}")


Configuration loaded.
  Audio      : ./content/audio.mp3 (source=509ba629751d)
  Whisper    : deepdml/faster-whisper-large-v3-turbo-ct2 | backend=faster-whisper | lang=en
  Whisper HW : device=cuda, compute=float16, isolated=True, mode=sequential, batch=4, beam=1
  Chunk      : 45s (overlap=5s)
  CLAP       : laion/larger_clap_general | segment=10s (overlap=2s)
  Embeddings : BAAI/bge-m3
  Retrieval  : k=10, reranker_top_n=5
  Generator  : mistral-nemo:latest (ollama)
  Qdrant     : http://localhost:6333
    Whisper collection : audio_whisper_509ba629751d
    CLAP collection    : audio_clap_509ba629751d


### 2. Audio Loading

Loads the MP3 and converts it to a 48 kHz mono waveform for CLAP audio embeddings.

Whisper transcription with the recommended `faster-whisper` backend reads `AUDIO_PATH`
directly, so it does not depend on this 48 kHz waveform.

> If the audio file is not yet available, you can download it with:
> ```bash
> pip install yt-dlp
> yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <YOUTUBE_URL>
> ```


In [25]:
import numpy as np
import librosa
from pathlib import Path

SAMPLE_RATE = 48000  # Hz — required by both Whisper and CLAP

def load_audio(path: str, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Load an audio file and return a normalised float32 mono waveform at `sr` Hz.
    Supports mp3, wav, flac, m4a via librosa/ffmpeg.
    """
    audio_path = Path(path)
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {path}\n"
            "Download with: yt-dlp -x --audio-format mp3 -o content/audio.mp3 <URL>"
        )
    waveform, _ = librosa.load(path, sr=sr, mono=True)
    print(f"Loaded: {audio_path.name}")
    print(f"  Duration : {len(waveform)/sr:.1f}s  ({len(waveform)/sr/60:.1f} min)")
    print(f"  Samples  : {len(waveform):,}  @ {sr} Hz")
    return waveform

waveform = load_audio(AUDIO_PATH)
AUDIO_DURATION_SECS = len(waveform) / SAMPLE_RATE


Loaded: audio.mp3
  Duration : 1192.1s  (19.9 min)
  Samples  : 57,220,992  @ 48000 Hz


---
## 3. Whisper Pipeline — Full-Text Conversion

```
MP3 → Whisper → word-level segments + timestamps
    → merge into text chunks (≤ AUDIO_CHUNK_SECS)
    → text embedding (EMBEDDING_MODEL)
    → Qdrant QDRANT_WHISPER_COLLECTION
    → BM25 on transcription
    → hybrid retrieval → LLM (text only)
```


### 3.1 Transcription with Whisper

```bash
pip install faster-whisper
```

The notebook still supports the Hugging Face backend by setting:

```env
WHISPER_BACKEND=hf
WHISPER_MODEL=openai/whisper-large-v3
```

For stability, faster-whisper runs in an isolated Python subprocess by default:

```env
WHISPER_SUBPROCESS=true
WHISPER_INFERENCE_MODE=sequential
```

The subprocess uses the same Python executable as the Jupyter kernel. If native
CTranslate2/CUDA inference fails, the kernel remains alive and the notebook
automatically retries sequential CPU `int8` inference. Disable that retry with
`WHISPER_SUBPROCESS_CPU_FALLBACK=false`.

`BatchedInferencePipeline` remains optional. After sequential transcription
succeeds, opt in with `WHISPER_INFERENCE_MODE=batched` and start with
`WHISPER_BATCH_SIZE=4`.


In [26]:
import torch

WHISPER_AVAILABLE = False
whisper_pipe = None
whisper_fw_model = None
whisper_fw_batched_model = None

def _resolve_whisper_device() -> str:
    """Resolve requested Whisper device, with a safe CPU fallback."""
    requested = (WHISPER_DEVICE or "auto").lower()
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        print("  [warn] WHISPER_DEVICE=cuda requested, but CUDA is not available. Falling back to CPU.")
        return "cpu"
    return requested

whisper_device = _resolve_whisper_device()
print(f"Loading Whisper model: {WHISPER_MODEL} ({WHISPER_BACKEND}) ...")

if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
    try:
        if WHISPER_INFERENCE_MODE not in {"sequential", "batched"}:
            raise ValueError(
                "WHISPER_INFERENCE_MODE must be 'sequential' or 'batched'. "
                f"Got: {WHISPER_INFERENCE_MODE!r}"
            )

        if WHISPER_SUBPROCESS:
            # Importing and constructing CTranslate2 objects is intentionally
            # deferred to a child process so Jupyter survives native failures.
            WHISPER_AVAILABLE = True
            print(
                f"  faster-whisper configured for isolated subprocess inference "
                f"on {whisper_device} (compute={WHISPER_COMPUTE_TYPE}, "
                f"mode={WHISPER_INFERENCE_MODE})"
            )
        else:
            from faster_whisper import WhisperModel

            whisper_fw_model = WhisperModel(
                WHISPER_MODEL,
                device=whisper_device,
                compute_type=WHISPER_COMPUTE_TYPE,
            )
            if WHISPER_INFERENCE_MODE == "batched":
                from faster_whisper import BatchedInferencePipeline
                whisper_fw_batched_model = BatchedInferencePipeline(model=whisper_fw_model)

            WHISPER_AVAILABLE = True
            print(
                f"  faster-whisper ready in the kernel on {whisper_device} "
                f"(compute={WHISPER_COMPUTE_TYPE}, mode={WHISPER_INFERENCE_MODE}, "
                f"batch={WHISPER_BATCH_SIZE if WHISPER_INFERENCE_MODE == 'batched' else 'n/a'})"
            )
    except Exception as e:
        print(f"  Failed to configure faster-whisper: {e}")
        print("    Install with: pip install faster-whisper")
        WHISPER_AVAILABLE = False

elif WHISPER_BACKEND.lower() in {"hf", "transformers", "huggingface"}:
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

    torch_dt = torch.float16 if whisper_device == "cuda" else torch.float32
    try:
        whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
        whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            WHISPER_MODEL,
            torch_dtype=torch_dt,
            low_cpu_mem_usage=True,
        ).to(whisper_device)

        whisper_pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=whisper_model,
            tokenizer=whisper_processor.tokenizer,
            feature_extractor=whisper_processor.feature_extractor,
            torch_dtype=torch_dt,
            device=0 if whisper_device == "cuda" else -1,
            return_timestamps=True,
            chunk_length_s=30,
            batch_size=WHISPER_BATCH_SIZE,
            generate_kwargs={"language": WHISPER_LANGUAGE} if WHISPER_LANGUAGE else {},
        )
        WHISPER_AVAILABLE = True
        print(f"  ✓ Hugging Face Whisper ready on {whisper_device}")
    except Exception as e:
        print(f"  ✗ Failed to load Hugging Face Whisper: {e}")
        WHISPER_AVAILABLE = False

else:
    raise ValueError(
        "Unsupported WHISPER_BACKEND. Use 'faster-whisper' or 'hf'. "
        f"Got: {WHISPER_BACKEND!r}"
    )


Loading Whisper model: deepdml/faster-whisper-large-v3-turbo-ct2 (faster-whisper) ...
  faster-whisper configured for isolated subprocess inference on cuda (compute=float16, mode=sequential)


In [27]:
import importlib, json, hashlib, os, re, subprocess, sys
from pathlib import Path

# ── Transcription cache ───────────────────────────────────────────────────────
# Whisper on a long audio file is slow — cache to disk to avoid re-running.
def _audio_hash(path: str) -> str:
    return _audio_source_id(path)

def _safe_cache_part(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-")

_cache_name = "_".join([
    "transcript",
    _audio_hash(AUDIO_PATH),
    _safe_cache_part(WHISPER_BACKEND),
    _safe_cache_part(WHISPER_MODEL.split("/")[-1]),
    _safe_cache_part(WHISPER_COMPUTE_TYPE),
    _safe_cache_part(WHISPER_INFERENCE_MODE),
    _safe_cache_part("subprocess" if WHISPER_SUBPROCESS else "inprocess"),
]) + ".json"
TRANSCRIPT_CACHE = Path(PERSIST_DIR) / _cache_name


def _faster_whisper_helper_path() -> Path:
    """Locate the subprocess helper from the pipeline or repository root."""
    candidates = [
        Path("transcribe_faster_whisper.py"),
        Path("audio_pipeline") / "transcribe_faster_whisper.py",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot find transcribe_faster_whisper.py. Run the notebook from the "
        "audio_pipeline folder or from the repository root."
    )


def _nvidia_library_paths() -> list[str]:
    """Return pip-installed NVIDIA library folders needed by CTranslate2."""
    paths = []
    for module_name in ("nvidia.cublas.lib", "nvidia.cudnn.lib"):
        try:
            module = importlib.import_module(module_name)
            module_file = getattr(module, "__file__", None)
            if module_file:
                paths.append(str(Path(module_file).resolve().parent))
        except ImportError:
            pass
    return paths


def _run_faster_whisper_subprocess(
    audio_path: str,
    cache_path: Path,
    *,
    device: str,
    compute_type: str,
    inference_mode: str,
) -> bool:
    """Launch native inference in a child process and report whether it succeeded."""
    command = [
        sys.executable,
        str(_faster_whisper_helper_path()),
        "--audio", str(Path(audio_path).resolve()),
        "--output", str(cache_path.resolve()),
        "--model", WHISPER_MODEL,
        "--device", device,
        "--compute-type", compute_type,
        "--language", WHISPER_LANGUAGE or "",
        "--inference-mode", inference_mode,
        "--batch-size", str(WHISPER_BATCH_SIZE),
        "--beam-size", str(WHISPER_BEAM_SIZE),
        "--vad-filter", str(WHISPER_VAD_FILTER).lower(),
    ]
    print(
        f"  Starting isolated faster-whisper: device={device}, "
        f"compute={compute_type}, mode={inference_mode}"
    )
    child_env = os.environ.copy()
    nvidia_paths = _nvidia_library_paths()
    if nvidia_paths:
        existing_library_path = child_env.get("LD_LIBRARY_PATH", "")
        child_env["LD_LIBRARY_PATH"] = os.pathsep.join(
            [*nvidia_paths, *([existing_library_path] if existing_library_path else [])]
        )
        print("  Added pip-installed NVIDIA libraries to child LD_LIBRARY_PATH.")
    completed = subprocess.run(command, check=False, env=child_env)
    if completed.returncode == 0 and cache_path.exists():
        return True
    print(f"  [warn] isolated faster-whisper exited with code {completed.returncode}.")
    return False


def _transcribe_faster_whisper_subprocess(audio_path: str, cache_path: Path) -> dict:
    """Transcribe outside the kernel, retrying safely on CPU when requested."""
    attempts = [(whisper_device, WHISPER_COMPUTE_TYPE, WHISPER_INFERENCE_MODE)]
    if WHISPER_SUBPROCESS_CPU_FALLBACK and whisper_device != "cpu":
        attempts.append(("cpu", "int8", "sequential"))

    for device, compute_type, inference_mode in attempts:
        if _run_faster_whisper_subprocess(
            audio_path,
            cache_path,
            device=device,
            compute_type=compute_type,
            inference_mode=inference_mode,
        ):
            with open(cache_path, encoding="utf-8") as transcript_file:
                return json.load(transcript_file)
        if device != "cpu" and len(attempts) > 1:
            print("  Retrying transcription with isolated CPU int8 inference...")

    raise RuntimeError(
        "Isolated faster-whisper transcription failed. The Jupyter kernel is "
        "still healthy. Review the child-process output above. For a manual "
        "CPU run set WHISPER_DEVICE=cpu and WHISPER_COMPUTE_TYPE=int8."
    )


def _transcribe_faster_whisper_inprocess(audio_path: str) -> dict:
    """Optional in-kernel path for environments known to have stable CTranslate2."""
    language = WHISPER_LANGUAGE or None
    common_kwargs = {
        "language": language,
        "vad_filter": WHISPER_VAD_FILTER,
        "beam_size": WHISPER_BEAM_SIZE,
        "word_timestamps": False,
    }
    if WHISPER_INFERENCE_MODE == "batched":
        if whisper_fw_batched_model is None:
            raise RuntimeError("Batched faster-whisper pipeline was not loaded.")
        segments_iter, info = whisper_fw_batched_model.transcribe(
            audio_path,
            batch_size=WHISPER_BATCH_SIZE,
            **common_kwargs,
        )
    else:
        segments_iter, info = whisper_fw_model.transcribe(audio_path, **common_kwargs)

    chunks = []
    texts = []
    for seg in segments_iter:
        text = seg.text.strip()
        if not text:
            continue
        chunks.append({"text": text, "timestamp": [float(seg.start), float(seg.end)]})
        texts.append(text)

    detected_language = getattr(info, "language", None)
    language_probability = getattr(info, "language_probability", None)
    if detected_language:
        probability_label = (
            f"{language_probability:.2f}" if language_probability is not None else "unknown"
        )
        print(f"  Detected language: {detected_language} ({probability_label})")

    return {
        "text": " ".join(texts).strip(),
        "chunks": chunks,
        "metadata": {
            "backend": "faster-whisper",
            "model": WHISPER_MODEL,
            "device": whisper_device,
            "compute_type": WHISPER_COMPUTE_TYPE,
            "inference_mode": WHISPER_INFERENCE_MODE,
            "batch_size": WHISPER_BATCH_SIZE if WHISPER_INFERENCE_MODE == "batched" else None,
            "beam_size": WHISPER_BEAM_SIZE,
            "vad_filter": WHISPER_VAD_FILTER,
            "detected_language": detected_language,
            "language_probability": language_probability,
            "isolated_subprocess": False,
        },
    }


def _transcribe_hf(waveform: np.ndarray) -> dict:
    """Transcribe waveform with the Hugging Face ASR pipeline."""
    result = whisper_pipe(waveform.copy(), return_timestamps=True)
    return {
        "text": result["text"],
        "chunks": [
            {"text": c["text"].strip(), "timestamp": list(c["timestamp"])}
            for c in result.get("chunks", [])
            if c.get("text", "").strip()
        ],
        "metadata": {
            "backend": "hf",
            "model": WHISPER_MODEL,
            "device": whisper_device,
        },
    }


def transcribe(waveform: np.ndarray, cache_path: Path, audio_path: str = AUDIO_PATH) -> dict:
    """Transcribe audio and return a common dict with 'text' and timestamped 'chunks'."""
    if cache_path.exists():
        print(f"Loading transcription from cache: {cache_path.name}")
        with open(cache_path, encoding="utf-8") as transcript_file:
            return json.load(transcript_file)

    if not WHISPER_AVAILABLE:
        raise RuntimeError("Whisper model not loaded.")

    print("Transcribing audio...")
    if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
        if WHISPER_SUBPROCESS:
            serialisable = _transcribe_faster_whisper_subprocess(audio_path, cache_path)
        else:
            serialisable = _transcribe_faster_whisper_inprocess(audio_path)
    else:
        serialisable = _transcribe_hf(waveform)

    if not cache_path.exists():
        with open(cache_path, "w", encoding="utf-8") as transcript_file:
            json.dump(serialisable, transcript_file, ensure_ascii=False, indent=2)
    print(f"  ✓ Transcription saved to cache: {cache_path.name}")
    return serialisable


transcription = transcribe(waveform, TRANSCRIPT_CACHE, AUDIO_PATH)
full_text = transcription["text"]
segments  = transcription["chunks"]   # list of {text, timestamp: [start, end]}

print(f"\nTranscription complete: {len(segments)} segments, {len(full_text)} chars")
print("\nFirst 500 chars:")
print(full_text[:500])


Loading transcription from cache: transcript_509ba629751d_faster-whisper_faster-whisper-large-v3-turbo-ct2_float16_sequential_subprocess.json

Transcription complete: 206 segments, 15948 chars

First 500 chars:
Attention is all you need. By Ashish Vaswani, Noam Shazir, Nicky Parmer, Jacob Uskarite, Lyon Jones, Aidan N. Gomez, Lukash Kaiser, and Ilya Polisukin. Abstract. The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the transformer, based solely on attention mechanis


### 3.2 Text Chunking with Timestamps


In [28]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class AudioChunk:
    """
    A chunk of transcribed audio with timestamp metadata.
    doc_type is always 'text' for both pipelines — the LLM only receives text.
    """
    content:    str            # transcribed text of this chunk
    doc_type:   str = "text"
    start_sec:  float = 0.0   # start time in the original audio
    end_sec:    float = 0.0   # end time in the original audio
    source_file: str = ""
    metadata:   dict = field(default_factory=dict)

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"


def merge_segments_into_chunks(
    segments:     List[dict],
    max_secs:     int = AUDIO_CHUNK_SECS,
    overlap_secs: int = AUDIO_CHUNK_OVERLAP_SECS,
    source_file:  str = AUDIO_PATH,
) -> List[AudioChunk]:
    """
    Merge consecutive Whisper segments into chunks of at most `max_secs` seconds.
    Adds `overlap_secs` of context from the previous chunk to each new chunk.

    Each Whisper segment has: {"text": str, "timestamp": [start, end]}
    End may be None for the last segment — use audio duration as fallback.
    """
    if not segments:
        return []

    chunks:  List[AudioChunk] = []
    buf_text:  List[str]   = []
    buf_segs:  List[dict]  = []
    buf_start: float       = segments[0]["timestamp"][0] or 0.0

    def flush(buf_text, buf_segs, buf_start):
        if not buf_text:
            return
        text  = " ".join(buf_text).strip()
        end   = buf_segs[-1]["timestamp"][1] or AUDIO_DURATION_SECS
        chunks.append(AudioChunk(
            content    = text,
            start_sec  = buf_start,
            end_sec    = end,
            source_file= source_file,
        ))

    overlap_buf: List[dict] = []   # segments carried over for context

    for seg in segments:
        ts    = seg["timestamp"]
        start = ts[0] if ts[0] is not None else (buf_start if buf_segs else 0.0)
        end   = ts[1] if ts[1] is not None else AUDIO_DURATION_SECS

        # Start a new chunk if max duration is exceeded
        if buf_segs and (end - buf_start) > max_secs:
            flush(buf_text, buf_segs, buf_start)
            # Carry-over overlap segments
            overlap_buf = [s for s in buf_segs if (s["timestamp"][1] or AUDIO_DURATION_SECS) >= (buf_start + max_secs - overlap_secs)]
            buf_text  = [s["text"] for s in overlap_buf]
            buf_segs  = list(overlap_buf)
            buf_start = overlap_buf[0]["timestamp"][0] if overlap_buf else start

        buf_text.append(seg["text"])
        buf_segs.append(seg)

    flush(buf_text, buf_segs, buf_start)
    return chunks


whisper_chunks = merge_segments_into_chunks(segments)
print(f"Created {len(whisper_chunks)} text chunks from {len(segments)} Whisper segments.")
print(f"  avg duration : {sum(c.end_sec - c.start_sec for c in whisper_chunks)/len(whisper_chunks):.1f}s")
print(f"\nFirst chunk {whisper_chunks[0].timestamp_label}:")
print(whisper_chunks[0].content[:300])


Created 29 text chunks from 206 Whisper segments.
  avg duration : 41.2s

First chunk [00:00 – 00:44]:
Attention is all you need. By Ashish Vaswani, Noam Shazir, Nicky Parmer, Jacob Uskarite, Lyon Jones, Aidan N. Gomez, Lukash Kaiser, and Ilya Polisukin. Abstract. The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a de


### 3.3 Text Embedding + Qdrant


In [29]:
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# ── Text embedding model ──────────────────────────────────────────────────────
print(f"Loading text embedding model: {EMBEDDING_MODEL} ...")
text_embedding = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {"trust_remote_code": True},
    encode_kwargs = {"normalize_embeddings": True},
)
EMBEDDING_DIM = len(text_embedding.embed_query("dim probe"))
print(f"  ✓ Text embedding ready (dim={EMBEDDING_DIM})")

# ── Qdrant client (reusable across both pipelines) ────────────────────────────
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

# ── Whisper collection ────────────────────────────────────────────────────────
if RESET_WHISPER_COLLECTION:
    client.delete_collection(QDRANT_WHISPER_COLLECTION)
    print(f"✓ Deleted '{QDRANT_WHISPER_COLLECTION}'")

if not client.collection_exists(QDRANT_WHISPER_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_WHISPER_COLLECTION,
        vectors_config  = VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created Whisper collection '{QDRANT_WHISPER_COLLECTION}' (dim={EMBEDDING_DIM})")
else:
    print(f"✓ Using existing Whisper collection '{QDRANT_WHISPER_COLLECTION}'")

whisper_vector_store = QdrantVectorStore(
    client          = client,
    collection_name = QDRANT_WHISPER_COLLECTION,
    embedding       = text_embedding,
)

# ── Docstore: UUID → AudioChunk ──────────────────────────────────────────────
whisper_docstore: Dict[str, AudioChunk] = {}


Loading text embedding model: BAAI/bge-m3 ...
  ✓ Text embedding ready (dim=1024)
✓ Using existing Whisper collection 'audio_whisper_509ba629751d'


In [30]:
from langchain_core.documents import Document

def index_whisper_chunks(
    chunks:       List[AudioChunk],
    vector_store,
    docstore:     dict,
    batch_size:   int = 64,
) -> List[str]:
    """Index transcribed text chunks into Qdrant Whisper collection."""
    all_ids: List[str] = []
    lc_docs: List[Document] = []

    for chunk in chunks:
        if not chunk.content.strip():
            continue
        uid = str(uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{AUDIO_SOURCE_ID}:whisper:{chunk.start_sec:.3f}:{chunk.end_sec:.3f}",
        ))
        all_ids.append(uid)
        docstore[uid] = chunk
        lc_docs.append(Document(
            page_content = chunk.content,
            metadata     = {
                "doc_id":    uid,
                "doc_type":  "text",
                "start_sec": chunk.start_sec,
                "end_sec":   chunk.end_sec,
                "timestamp": chunk.timestamp_label,
                "source":    chunk.source_file,
            },
        ))

    print(f"Indexing {len(lc_docs)} chunks into '{QDRANT_WHISPER_COLLECTION}' ...")
    for i in range(0, len(lc_docs), batch_size):
        batch = lc_docs[i : i + batch_size]
        try:
            vector_store.add_documents(batch, ids=all_ids[i : i + batch_size])
        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e}")
        print(f"  [{min(i+batch_size, len(lc_docs))}/{len(lc_docs)}] inserted")

    print(f"✓ Whisper indexing complete: {len(all_ids)} chunks.")
    return all_ids

whisper_ids = index_whisper_chunks(whisper_chunks, whisper_vector_store, whisper_docstore)


Indexing 29 chunks into 'audio_whisper_509ba629751d' ...
  [29/29] inserted
✓ Whisper indexing complete: 29 chunks.


### 3.4 BM25 on Transcription


In [31]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import Tuple

class AudioBM25Index:
    """BM25 index over AudioChunk content."""
    def __init__(self):
        self._chunks: List[AudioChunk] = []
        self._bm25                     = None

    def build(self, chunks: List[AudioChunk]) -> None:
        self._chunks  = chunks
        tokenized     = [c.content.lower().split() for c in chunks]
        self._bm25    = BM25Okapi(tokenized)
        print(f"BM25 built on {len(chunks)} Whisper chunks.")

    def retrieve(self, query: str, top_k: int = RETRIEVER_K) -> List[Tuple[float, AudioChunk]]:
        if self._bm25 is None or not self._chunks:
            return []
        scores  = self._bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self._chunks[i]) for i in top_idx if scores[i] > 0]

whisper_bm25 = AudioBM25Index()
if ENABLE_BM25:
    whisper_bm25.build(whisper_chunks)
else:
    print("BM25 disabled.")


BM25 built on 29 Whisper chunks.


### 3.5 Whisper Hybrid Retrieval


In [32]:
from sentence_transformers import CrossEncoder

# ── Cross-encoder reranker ─────────────────────────────────────────────────
if ENABLE_RERANKING:
    print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
    try:
        reranker           = CrossEncoder(RERANKER_MODEL)
        RERANKER_AVAILABLE = True
        print("  ✓ Reranker ready.")
    except Exception as e:
        print(f"  ✗ {e}")
        reranker           = None
        RERANKER_AVAILABLE = False
else:
    reranker           = None
    RERANKER_AVAILABLE = False
    print("Reranking disabled.")


def whisper_hybrid_retrieve(query: str) -> List[AudioChunk]:
    """
    Hybrid retrieval on the Whisper pipeline:
      1. Dense: text query → text embedding → Qdrant Whisper collection
      2. BM25 : keyword matching on raw transcription chunks
      3. Deduplication by content
      4. Cross-encoder reranking on merged candidates
    """
    seen:       set                     = set()
    candidates: List[Tuple[str, AudioChunk]] = []   # (snippet, chunk)

    # Dense
    try:
        dense_hits = whisper_vector_store.similarity_search(query, k=RETRIEVER_K)
        for lc in dense_hits:
            uid   = lc.metadata.get("doc_id")
            chunk = whisper_docstore.get(uid) if uid else None
            if chunk is None:
                chunk = AudioChunk(content=lc.page_content,
                                   start_sec=lc.metadata.get("start_sec", 0),
                                   end_sec=lc.metadata.get("end_sec", 0))
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((lc.page_content, chunk))
    except Exception as e:
        print(f"[whisper_retrieve] Dense error: {e}")

    # BM25
    if ENABLE_BM25:
        for score, chunk in whisper_bm25.retrieve(query, top_k=RETRIEVER_K):
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((chunk.content[:500], chunk))

    if not candidates:
        return []

    # Reranking
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, s) for s, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [c for _, c in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [c for _, c in ranked[:RERANKER_TOP_N]]
    else:
        final  = [c for _, c in candidates[:RERANKER_TOP_N]]

    print(f"[whisper_retrieve] {len(candidates)} candidates → {len(final)} "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_w = whisper_hybrid_retrieve("What is positional encoding and why is it necessary in the Transformer?")
print(f"\nRetrieved {len(test_w)} chunks:")
for i, c in enumerate(test_w):
    print(f"  [{i}] {c.timestamp_label}  {c.content[:80]!r}")


Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...
  ✓ Reranker ready.
[whisper_retrieve] 15 candidates → 5 (query: 'What is positional encoding and why is it necessary in ')

Retrieved 5 chunks:
  [0] [14:29 – 15:04]  'In this work, we use sine and cosine functions of different frequencies. The pos'
  [1] [15:04 – 15:39]  'We chose this function because we hypothesized it would allow the model to easil'
  [2] [13:51 – 14:33]  'In embedding layers, we multiply those weights by the square root of D model. 3.'
  [3] [03:51 – 04:35]  'In these models, the number of operations required to relate signals from two ar'
  [4] [05:47 – 06:32]  'At each step, the model is auto-regressive, consuming the previously generated s'


---
## 4. CLAP Pipeline — Shared Audio-Text Semantic Space

```
MP3 → split into fixed-length segments (CLAP_SEGMENT_SECS)
    → CLAP audio encoder → audio embeddings
    → Qdrant QDRANT_CLAP_COLLECTION

Query (text) → CLAP text encoder → similarity search → top-k audio segments
    → resolve segment timestamps → extract matching Whisper transcription
    → LLM (text only)
```

The CLAP text encoder and audio encoder share the same vector space,
enabling cross-modal retrieval: a text query finds audio segments by
semantic similarity without requiring keyword overlap in the transcription.


### 4.1 CLAP Model


In [33]:
import inspect
from transformers import ClapModel, ClapProcessor
import torch
import numpy as np

print(f"Loading CLAP model: {CLAP_MODEL} ...")
clap_device = "cuda" if torch.cuda.is_available() else "cpu"

def clap_processor_call(*, audio=None, text=None, **kwargs):
    """Call ClapProcessor with the audio keyword expected by this transformers build."""
    signature = inspect.signature(clap_processor.__call__)
    if "audios" in signature.parameters:
        return clap_processor(audios=audio, text=text, **kwargs)
    return clap_processor(audio=audio, text=text, **kwargs)


def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out

    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output

    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state.mean(dim=1)

    if isinstance(out, (tuple, list)):
        return out[0]

    raise TypeError(f"Unsupported CLAP output type: {type(out)}")

try:
    clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL)

    clap_model = ClapModel.from_pretrained(
        CLAP_MODEL,
        torch_dtype=torch.float32,   # <- cambia qui
    ).to(clap_device)

    clap_model.eval()

    with torch.no_grad():
        dummy_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)
        dummy_inputs = clap_processor_call(
            audio=dummy_audio,
            return_tensors="pt",
            sampling_rate=SAMPLE_RATE,
        )

        dummy_inputs = {k: v.to(clap_device) for k, v in dummy_inputs.items()}

        out = clap_model.get_audio_features(**dummy_inputs)

        audio_emb = clap_output_to_tensor(out)
        audio_emb = torch.nn.functional.normalize(audio_emb, dim=-1)

        CLAP_DIM = audio_emb.shape[-1]

    CLAP_AVAILABLE = True
    print(f"  ✓ CLAP ready on {clap_device} | dim={CLAP_DIM}")

except Exception as e:
    print(f"  ✗ Could not load CLAP: {e}")
    clap_model = clap_processor = None
    CLAP_AVAILABLE = False
    CLAP_DIM = 512


Loading CLAP model: laion/larger_clap_general ...
  ✓ CLAP ready on cuda | dim=512


### 4.2 Audio Segmentation and CLAP Embedding


In [34]:
@dataclass
class AudioSegment:
    """A fixed-length audio segment with timestamp and precomputed CLAP embedding."""
    start_sec:   float
    end_sec:     float
    source_file: str
    transcript:  str = ""    # Whisper text for this time range (populated later)
    doc_type:    str = "text"  # always "text" — LLM receives transcript, not audio

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        """Alias for compatibility with metric functions that expect .content."""
        return self.transcript

def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "pooler_output"):
        return out.pooler_output
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state.mean(dim=1)
    raise TypeError(f"Unexpected CLAP output type: {type(out)}")

def split_waveform_into_segments(
    waveform:     np.ndarray,
    sr:           int   = SAMPLE_RATE,
    seg_secs:     int   = CLAP_SEGMENT_SECS,
    overlap_secs: int   = CLAP_SEGMENT_OVERLAP,
    source_file:  str   = AUDIO_PATH,
) -> List[AudioSegment]:
    """Split a waveform into overlapping fixed-length AudioSegments."""
    step   = (seg_secs - overlap_secs) * sr
    length = seg_secs * sr
    segs   = []
    pos    = 0
    while pos < len(waveform):
        end_sample = min(pos + length, len(waveform))
        start_s    = pos / sr
        end_s      = end_sample / sr
        segs.append(AudioSegment(start_sec=start_s, end_sec=end_s, source_file=source_file))
        if end_sample == len(waveform):
            break
        pos += step
    return segs


def embed_audio_segments_clap(
    waveform:  np.ndarray,
    segments:  List[AudioSegment],
    batch_size: int = 8,
) -> np.ndarray:
    """
    Embed each AudioSegment with the CLAP audio encoder.
    Returns (N, CLAP_DIM) float32 array, L2-normalised.
    """
    if not CLAP_AVAILABLE:
        return np.zeros((len(segments), CLAP_DIM), dtype=np.float32)

    all_vecs = []
    total    = len(segments)
    print(f"Embedding {total} audio segments with CLAP ...")

    for i in range(0, total, batch_size):
        batch_segs = segments[i : i + batch_size]
        batch_audio = []
        for seg in batch_segs:
            s = int(seg.start_sec * SAMPLE_RATE)
            e = int(seg.end_sec   * SAMPLE_RATE)
            batch_audio.append(waveform[s:e].astype(np.float32))

        try:
            inputs = clap_processor_call(
                audio=batch_audio,
                return_tensors="pt",
                sampling_rate=SAMPLE_RATE,
                padding=True,
            )

            inputs = {k: v.to(clap_device) for k, v in inputs.items()}

            with torch.no_grad():
                out = clap_model.get_audio_features(**inputs)
                vecs = clap_output_to_tensor(out)
                vecs = torch.nn.functional.normalize(vecs, dim=-1).float()

            all_vecs.append(vecs.cpu().numpy())

        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e} — using zero vectors")
            all_vecs.append(np.zeros((len(batch_segs), CLAP_DIM), dtype=np.float32))

        print(f"  [{min(i+batch_size, total)}/{total}] embedded")

    return np.vstack(all_vecs)


# Segment + embed
clap_segments = split_waveform_into_segments(waveform)
clap_vectors  = embed_audio_segments_clap(waveform, clap_segments)

print(f"\nSegments : {len(clap_segments)}")
print(f"Avg dur  : {sum(s.end_sec-s.start_sec for s in clap_segments)/len(clap_segments):.1f}s")
print(f"Vectors  : {clap_vectors.shape}")


Embedding 149 audio segments with CLAP ...
  [8/149] embedded
  [16/149] embedded
  [24/149] embedded
  [32/149] embedded
  [40/149] embedded
  [48/149] embedded
  [56/149] embedded
  [64/149] embedded
  [72/149] embedded
  [80/149] embedded
  [88/149] embedded
  [96/149] embedded
  [104/149] embedded
  [112/149] embedded
  [120/149] embedded
  [128/149] embedded
  [136/149] embedded
  [144/149] embedded
  [149/149] embedded

Segments : 149
Avg dur  : 10.0s
Vectors  : (149, 512)


### 4.3 Attach Whisper Transcription to CLAP Segments


In [35]:
def get_transcript_for_range(
    segments: List[dict],
    start_s:  float,
    end_s:    float,
    padding:  float = 1.0,
) -> str:
    """
    Extract the Whisper transcription covering [start_s - padding, end_s + padding].
    Uses the segment-level timestamps already available from the transcription.
    """
    relevant = []
    for seg in segments:
        ts_start = seg["timestamp"][0] or 0.0
        ts_end   = seg["timestamp"][1] or AUDIO_DURATION_SECS
        # Include segment if it overlaps with the query window
        if ts_end >= (start_s - padding) and ts_start <= (end_s + padding):
            relevant.append(seg["text"])
    return " ".join(relevant).strip()


# Attach transcript to every CLAP segment
print("Attaching Whisper transcription to CLAP segments ...")
for seg in clap_segments:
    seg.transcript = get_transcript_for_range(segments, seg.start_sec, seg.end_sec)

# How many segments have non-empty transcription?
with_text = sum(1 for s in clap_segments if s.transcript.strip())
print(f"✓ {with_text}/{len(clap_segments)} segments have transcription coverage.")

# Preview
print(f"\nFirst CLAP segment {clap_segments[0].timestamp_label}:")
print(f"  Transcript: {clap_segments[0].transcript[:200]!r}")


Attaching Whisper transcription to CLAP segments ...
✓ 132/149 segments have transcription coverage.

First CLAP segment [00:00 – 00:10]:
  Transcript: 'Attention is all you need. By Ashish Vaswani, Noam Shazir, Nicky Parmer, Jacob Uskarite, Lyon Jones, Aidan N. Gomez, Lukash Kaiser, and Ilya Polisukin.'


### 4.4 CLAP Qdrant Collection + Indexing


In [36]:
from qdrant_client.http.models import PointStruct

# ── CLAP collection ───────────────────────────────────────────────────────────
if RESET_CLAP_COLLECTION and client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.delete_collection(QDRANT_CLAP_COLLECTION)
    print(f"✓ Deleted '{QDRANT_CLAP_COLLECTION}'")

if not client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_CLAP_COLLECTION,
        vectors_config  = VectorParams(size=CLAP_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created CLAP collection '{QDRANT_CLAP_COLLECTION}' (dim={CLAP_DIM})")
else:
    print(f"✓ Using existing CLAP collection '{QDRANT_CLAP_COLLECTION}'")

# Docstore: UUID → AudioSegment
clap_docstore: Dict[str, AudioSegment] = {}

# ── Index all segments ────────────────────────────────────────────────────────
UPSERT_BATCH = 64
clap_ids     = []
points       = []

for seg, vec in zip(clap_segments, clap_vectors):
    uid = str(uuid.uuid5(
        uuid.NAMESPACE_URL,
        f"{AUDIO_SOURCE_ID}:clap:{seg.start_sec:.3f}:{seg.end_sec:.3f}",
    ))
    clap_ids.append(uid)
    clap_docstore[uid] = seg
    points.append(PointStruct(
        id      = uid,
        vector  = vec.tolist(),
        payload = {
            "start_sec": seg.start_sec,
            "end_sec":   seg.end_sec,
            "timestamp": seg.timestamp_label,
            "source":    seg.source_file,
            "preview":   seg.transcript[:100],
        },
    ))

print(f"Inserting {len(points)} CLAP vectors into '{QDRANT_CLAP_COLLECTION}' ...")
inserted = 0
for i in range(0, len(points), UPSERT_BATCH):
    batch = points[i : i + UPSERT_BATCH]
    try:
        client.upsert(collection_name=QDRANT_CLAP_COLLECTION, points=batch)
        inserted += len(batch)
    except Exception as e:
        print(f"  ✗ Batch {i//UPSERT_BATCH}: {e}")

print(f"✓ CLAP indexing complete: {inserted}/{len(points)} segments.")


✓ Using existing CLAP collection 'audio_clap_509ba629751d'
Inserting 149 CLAP vectors into 'audio_clap_509ba629751d' ...
✓ CLAP indexing complete: 149/149 segments.


### 4.5 CLAP Hybrid Retrieval


In [37]:
def encode_query_clap(query: str) -> np.ndarray:
    """
    Encode a text query with the CLAP text encoder.
    Returns a normalised float32 vector of shape (CLAP_DIM,).
    The CLAP text and audio encoders share the same space:
    text queries can directly retrieve audio segments by cosine similarity.
    """
    if not CLAP_AVAILABLE:
        raise RuntimeError("CLAP model not available.")
    inputs = clap_processor(text=[query], return_tensors="pt", padding=True).to(clap_device)
    with torch.no_grad():
        vec = clap_model.get_text_features(**inputs)
        vec = clap_output_to_tensor(vec)
        vec = torch.nn.functional.normalize(vec, dim=-1).float()
    return vec.cpu().numpy()[0]


def clap_retrieve(query: str) -> List[AudioSegment]:
    """
    CLAP retrieval:
      1. Encode query with CLAP text encoder → query vector
      2. Cosine similarity search in QDRANT_CLAP_COLLECTION → audio segments
      3. Cross-encoder reranking on the *transcript* text of retrieved segments
         (CLAP retrieves by audio similarity; reranker refines by textual relevance)

    Note: BM25 is intentionally not used in this pipeline.
    CLAP embeddings already capture semantic audio content; keyword
    matching on the transcription would mix two retrieval signals
    that operate at different abstraction levels.
    """
    if not CLAP_AVAILABLE:
        print("[clap_retrieve] CLAP not available.")
        return []

    seen:        set                         = set()
    candidates:  List[Tuple[str, AudioSegment]] = []

    try:
        query_vec = encode_query_clap(query)
        result = client.query_points(
            collection_name = QDRANT_CLAP_COLLECTION,
            query           = query_vec.tolist(),
            limit           = RETRIEVER_K,
            with_payload    = True,
        )

        hits = result.points

        for h in hits:
            seg = clap_docstore.get(str(h.id))
            if seg is None or seg.content in seen:
                continue
            seen.add(seg.content)
            candidates.append((seg.transcript[:500], seg))
    except Exception as e:
        print(f"[clap_retrieve] Error: {e}")
        return []

    if not candidates:
        return []

    # Rerank on transcript text
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, t) for t, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [s for _, s in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [s for _, s in ranked[:RERANKER_TOP_N]]
    else:
        final  = [s for _, s in candidates[:RERANKER_TOP_N]]

    print(f"[clap_retrieve] {len(hits)} hits → {len(final)} after rerank "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_c = clap_retrieve("What is positional encoding and why is it necessary in the Transformer?")
print(f"\nRetrieved {len(test_c)} segments:")
for i, s in enumerate(test_c):
    print(f"  [{i}] {s.timestamp_label}  {s.transcript[:80]!r}")


[clap_retrieve] 10 hits → 5 after rerank (query: 'What is positional encoding and why is it necessary in ')

Retrieved 5 segments:
  [0] [12:40 – 12:50]  'In addition to attention sub-layers, each of the layers in our encoder and decod'
  [1] [05:04 – 05:14]  'To the best of our knowledge, however, the transformer is the first transduction'
  [2] [04:32 – 04:42]  'positions of a single sequence in order to compute a representation of the seque'
  [3] [03:52 – 04:02]  'In these models, the number of operations required to relate signals from two ar'
  [4] [09:52 – 10:02]  ''


---
## 5. Generation Model

Both pipelines pass **text only** to the LLM:
the Whisper pipeline passes transcribed chunk text,
the CLAP pipeline passes the Whisper transcript of the retrieved audio segment.


In [38]:
import gc, os, platform, subprocess
from urllib.request import urlopen

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_AVAILABLE     = False
USE_OLLAMA_GENERATOR    = False
gen_tokenizer           = None
gen_text_model          = None
ollama_llm              = None


def _int_env(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except (TypeError, ValueError):
        return default


OLLAMA_NUM_CTX = _int_env("OLLAMA_NUM_CTX", 4096)
OLLAMA_NUM_PREDICT = _int_env("OLLAMA_NUM_PREDICT", 512)
OLLAMA_KEEP_ALIVE = os.getenv("OLLAMA_KEEP_ALIVE", "0s")
OLLAMA_NUM_GPU = os.getenv("OLLAMA_NUM_GPU", "").strip()


def _ollama_runtime_kwargs() -> dict:
    kwargs = {
        "num_ctx": OLLAMA_NUM_CTX,
        "num_predict": OLLAMA_NUM_PREDICT,
        "keep_alive": OLLAMA_KEEP_ALIVE,
    }
    if OLLAMA_NUM_GPU:
        kwargs["num_gpu"] = int(OLLAMA_NUM_GPU)
    return kwargs


def _release_torch_cache() -> None:
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def _invoke_ollama_generator(prompt: str) -> str:
    _release_torch_cache()
    try:
        llm = globals().get("ollama_llm") or globals().get("_ollama_llm")
        if llm is None:
            return "[Ollama generator unavailable]"
        return llm.invoke(prompt).content.strip()
    except Exception as exc:
        message = str(exc)
        if "requires more system memory" in message.lower():
            return (
                "[Ollama unavailable: not enough system RAM to load the generator. "
                "Restart idle notebook kernels, close other models, lower OLLAMA_NUM_CTX, "
                "or choose a smaller GENERATION_MODEL.]"
            )
        raise
def _normalise_ollama_url(url: str) -> str:
    url = (url or "").strip().rstrip("/")
    if not url:
        return ""
    if "://" not in url:
        url = "http://" + url
    return url


def _is_wsl_runtime() -> bool:
    return bool(os.getenv("WSL_DISTRO_NAME")) or "microsoft" in platform.release().lower()


def _windows_host_ollama_url() -> str:
    if not _is_wsl_runtime():
        return ""
    try:
        win_host_ip = subprocess.check_output(
            "ip route | awk '/default/ {print $3; exit}'",
            shell=True,
            text=True,
        ).strip()
        return f"http://{win_host_ip}:11434" if win_host_ip else ""
    except Exception:
        return ""


def _candidate_ollama_urls() -> list[str]:
    candidates: list[str] = []

    explicit = _normalise_ollama_url(
        globals().get("OLLAMA_BASE_URL", "") or os.getenv("OLLAMA_BASE_URL", "")
    )
    if explicit and "0.0.0.0" not in explicit:
        candidates.append(explicit)

    wsl_host = _windows_host_ollama_url()
    if wsl_host:
        candidates.append(wsl_host)

    env_host = _normalise_ollama_url(os.getenv("OLLAMA_HOST", ""))
    if env_host and "0.0.0.0" not in env_host:
        candidates.append(env_host)

    if platform.system().lower().startswith("win"):
        candidates.extend(["http://127.0.0.1:11434", "http://localhost:11434"])
    else:
        candidates.extend(["http://127.0.0.1:11434", "http://localhost:11434"])

    unique: list[str] = []
    for url in candidates:
        url = _normalise_ollama_url(url)
        if url and url not in unique:
            unique.append(url)
    return unique


def _resolve_ollama_base_url(timeout: float = 2.0) -> str:
    errors = []
    for url in _candidate_ollama_urls():
        try:
            with urlopen(url + "/api/tags", timeout=timeout) as response:
                if 200 <= response.status < 300:
                    return url
        except Exception as exc:
            errors.append(f"{url} -> {type(exc).__name__}: {exc}")
    raise RuntimeError("No reachable Ollama endpoint. Tried:\n" + "\n".join(errors))


if GENERATION_BACKEND == "ollama":
    try:
        from langchain_ollama import ChatOllama

        OLLAMA_BASE_URL = _resolve_ollama_base_url()
        ollama_llm = ChatOllama(
            model=GENERATION_MODEL,
            base_url=OLLAMA_BASE_URL,
            temperature=0,
            **_ollama_runtime_kwargs(),
        )
        USE_OLLAMA_GENERATOR = True
        GENERATOR_AVAILABLE  = True
        print(f"✓ Ollama generator ready: {GENERATION_MODEL} at {OLLAMA_BASE_URL}")
    except Exception as e:
        print(f"✗ Ollama: {e}")
else:
    print(f"Loading HF generation model: {GENERATION_MODEL} ...")
    try:
        gen_tokenizer  = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
        gen_text_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL,
            torch_dtype = HF_TORCH_DTYPE,
            device_map  = HF_DEVICE_MAP,
            trust_remote_code = True,
        )
        gen_text_model.eval()
        GENERATOR_AVAILABLE = True
        print(f"  ✓ Generator ready: {GENERATION_MODEL}")
    except Exception as e:
        print(f"  ✗ {e}")


def generate_answer(retrieved: List, question: str) -> str:
    """
    Build a text-only context from retrieved AudioChunk / AudioSegment objects
    and generate an answer with the local model.
    Always text-only: audio content is represented via its Whisper transcript.
    """
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"

    context_parts = []
    for item in retrieved:
        ts   = getattr(item, "timestamp_label", "")
        text = item.content.strip()
        if text:
            context_parts.append(f"{ts}\n{text}")

    context_str = "\n\n".join(context_parts) if context_parts else "[No context retrieved]"
    prompt = (
        "Answer the question using only the provided context from an audio transcript. "
        "If the context is insufficient, say so explicitly.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {question}"
    )

    if USE_OLLAMA_GENERATOR:
        return _invoke_ollama_generator(prompt)

    messages = [{"role": "user", "content": prompt}]
    if hasattr(gen_tokenizer, "apply_chat_template"):
        text_in = gen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt

    inputs = gen_tokenizer([text_in], return_tensors="pt").to(gen_text_model.device)
    with torch.no_grad():
        generated = gen_text_model.generate(
            **inputs,
            max_new_tokens = GENERATION_MAX_NEW_TOKENS,
            do_sample      = False,
            temperature    = None,
            pad_token_id   = gen_tokenizer.eos_token_id,
        )
    new_tokens = generated[:, inputs.input_ids.shape[1]:]
    return gen_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


✓ Ollama generator ready: mistral-nemo:latest at http://127.0.0.1:11434


In [39]:
# Quick generation test — both pipelines
q = "Describe the overall architecture of the Transformer model"

print("── Whisper pipeline ─────────────────────────────────────────────────")
w_docs = whisper_hybrid_retrieve(q)
w_answer = generate_answer(w_docs, q)
print(w_answer)

print("\n── CLAP pipeline ────────────────────────────────────────────────────")
c_docs = clap_retrieve(q)
c_answer = generate_answer(c_docs, q)
print(c_answer)


── Whisper pipeline ─────────────────────────────────────────────────
[whisper_retrieve] 13 candidates → 5 (query: 'Describe the overall architecture of the Transformer mo')
The Transformer follows an encoder-decoder architecture for sequence transduction tasks. Here's its overall architecture:

1. **Encoder**:
   - Composed of a stack of six identical layers (N=6).
   - Each layer has two sub-layers:
     1. Multi-head self-attention mechanism.
     2. Position-wise, fully connected feed-forward network.
   - Residual connections are employed around each sub-layer, followed by layer normalization.

2. **Decoder**:
   - Similar to the encoder but with an additional multi-head attention sub-layer that attends to the encoder's output.
   - Also has two other sub-layers: multi-head self-attention and position-wise feed-forward network.
   - Residual connections are used around each sub-layer, followed by layer normalization.

3. **Auto-regressive Generation**:
   - The model generates sym

---
## 6. Evaluation

This section evaluates the audio RAG approaches with answer-level and retrieval-level metrics:

| Metric | What it checks |
|---|---|
| `BERTScore` | semantic similarity between generated and expected answer |
| `Precision` / `Recall` | lexical overlap of answer tokens against the expected answer |
| `Context Recall` | how many annotated claims are present in retrieved transcript context |
| `Timestamp Coverage` | how many inferred evidence time windows are covered by retrieved audio chunks/segments |
| `Must` / `Should` recall | claim coverage split by required vs desirable facts |

Run `6.4 Generate Evaluation Answers` once to cache retrieval, answers, and metrics. The widget after it only filters cached rows, so changing selectors does not regenerate answers.


### 6.1 Test Questions and Annotated Claims


In [40]:
# These questions are intentionally shared with a Transformer-paper RAG baseline.
# Every answer is also supported by the current audio narration, so timestamp
# coverage remains meaningful for the Whisper and CLAP pipelines.
TEST_SET = [
    {
        "question": "Describe the overall architecture of the Transformer model",
        "expected_answer":
            "The Transformer is an encoder-decoder sequence transduction model based "
            "entirely on attention. The encoder and decoder each contain a stack of six "
            "layers. Encoder layers contain multi-head self-attention and position-wise "
            "feed-forward sub-layers. Decoder layers add encoder-decoder attention and "
            "masked self-attention so predictions cannot depend on future output tokens. "
            "Residual connections and layer normalization surround the sub-layers.",
        "claims": [
            {"text": "The Transformer has an encoder-decoder architecture", "importance": "must", "source": "audio"},
            {"text": "The Transformer relies entirely on attention instead of recurrence or convolution", "importance": "must", "source": "audio"},
            {"text": "The encoder contains six identical layers", "importance": "must", "source": "audio"},
            {"text": "Encoder layers contain multi-head self-attention and feed-forward sub-layers", "importance": "must", "source": "audio"},
            {"text": "The decoder contains six identical layers and an additional encoder-decoder attention sub-layer", "importance": "should", "source": "audio"},
            {"text": "Residual connections and layer normalization are used", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": True,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Transformer overview", "start_sec": 326.16, "end_sec": 365.50, "timestamp": "[05:26 - 06:05]"},
            {"label": "Encoder and decoder stacks", "start_sec": 369.66, "end_sec": 455.88, "timestamp": "[06:09 - 07:35]"},
        ],
        "context_keywords": ["encoder", "decoder", "six", "self-attention", "feed-forward", "residual", "layer normalization"],
        "answer_keywords": ["encoder", "decoder", "six", "attention", "residual"],
    },
    {
        "question": "What BLEU score did the Transformer achieve on WMT 2014 English-to-German translation?",
        "expected_answer":
            "The Transformer achieved 28.4 BLEU on the WMT 2014 English-to-German "
            "translation task, improving over the previous best results, including "
            "ensembles, by more than 2 BLEU.",
        "claims": [
            {"text": "The model achieved 28.4 BLEU on WMT 2014 English-to-German translation", "importance": "must", "source": "audio"},
            {"text": "The result improved over previous best results including ensembles", "importance": "must", "source": "audio"},
            {"text": "The improvement was more than 2 BLEU", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": False,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "WMT 2014 English-to-German result", "start_sec": 49.86, "end_sec": 62.88, "timestamp": "[00:49 - 01:02]"},
        ],
        "context_keywords": ["28.4", "BLEU", "WMT", "2014", "English", "German", "ensembles"],
        "answer_keywords": ["28.4", "BLEU", "WMT 2014", "English-to-German"],
    },
    {
        "question": "Why does the Transformer use self-attention instead of recurrent or convolutional layers?",
        "expected_answer":
            "Self-attention reduces sequential computation, allows substantially more "
            "parallelization, and connects distant positions with shorter paths. This "
            "makes long-range dependencies easier to learn than with recurrent or "
            "convolutional layers. A self-attention layer connects all positions with "
            "a constant number of sequentially executed operations.",
        "claims": [
            {"text": "Self-attention reduces sequential computation", "importance": "must", "source": "audio"},
            {"text": "The Transformer allows substantially more parallelization", "importance": "must", "source": "audio"},
            {"text": "Shorter paths make long-range dependencies easier to learn", "importance": "must", "source": "audio"},
            {"text": "Self-attention connects all positions with a constant number of sequential operations", "importance": "should", "source": "audio"},
            {"text": "Recurrent computation is inherently sequential", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": True,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Sequential-computation limitation", "start_sec": 115.76, "end_sec": 165.42, "timestamp": "[01:55 - 02:45]"},
            {"label": "Transformer parallelization", "start_sec": 185.62, "end_sec": 211.66, "timestamp": "[03:05 - 03:31]"},
            {"label": "Self-attention comparison criteria", "start_sec": 939.80, "end_sec": 1033.68, "timestamp": "[15:39 - 17:13]"},
        ],
        "context_keywords": ["self-attention", "parallelization", "sequential", "long-range", "dependencies", "recurrent", "convolutional"],
        "answer_keywords": ["self-attention", "parallelization", "sequential", "long-range dependencies"],
    },
    {
        "question": "What is positional encoding and why is it necessary in the Transformer?",
        "expected_answer":
            "Positional encodings are added to the input embeddings at the bottoms of "
            "the encoder and decoder stacks to inject relative or absolute token-position "
            "information. They are needed because the Transformer contains no recurrence "
            "or convolution. The paper uses sine and cosine functions of different frequencies.",
        "claims": [
            {"text": "The Transformer contains no recurrence and no convolution", "importance": "must", "source": "audio"},
            {"text": "The model must inject relative or absolute token-position information", "importance": "must", "source": "audio"},
            {"text": "Positional encodings are added to the input embeddings", "importance": "must", "source": "audio"},
            {"text": "The paper uses sine and cosine functions of different frequencies", "importance": "should", "source": "audio"},
            {"text": "Positional encodings have the same dimension as embeddings so they can be summed", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": True,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Positional encoding motivation and construction", "start_sec": 838.64, "end_sec": 873.62, "timestamp": "[13:58 - 14:33]"},
        ],
        "context_keywords": ["positional", "encoding", "recurrence", "convolution", "position", "embeddings", "sine", "cosine"],
        "answer_keywords": ["positional encoding", "position", "embeddings", "sine", "cosine"],
    },
    {
        "question": "How does the decoder differ from the encoder in the Transformer?",
        "expected_answer":
            "Like the encoder, the decoder has a stack of six layers with residual "
            "connections and layer normalization. Each decoder layer adds a third "
            "sub-layer that performs multi-head attention over the encoder output. "
            "Its self-attention is masked so positions cannot attend to subsequent "
            "positions, preserving autoregressive generation.",
        "claims": [
            {"text": "The decoder contains six identical layers", "importance": "must", "source": "audio"},
            {"text": "The decoder adds a third sub-layer with multi-head attention over the encoder output", "importance": "must", "source": "audio"},
            {"text": "Decoder self-attention prevents positions from attending to subsequent positions", "importance": "must", "source": "audio"},
            {"text": "Masking ensures predictions depend only on known earlier outputs", "importance": "should", "source": "audio"},
            {"text": "Residual connections and layer normalization are used in the decoder", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": True,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Decoder stack and masking", "start_sec": 413.74, "end_sec": 455.88, "timestamp": "[06:53 - 07:35]"},
        ],
        "context_keywords": ["decoder", "six", "multi-head attention", "encoder output", "subsequent positions", "masking"],
        "answer_keywords": ["decoder", "masked", "encoder output", "subsequent positions"],
    },
    {
        "question": "Explain how multi-head attention works",
        "expected_answer":
            "Multi-head attention linearly projects queries, keys, and values multiple "
            "times with learned projections. Attention is applied to each projected "
            "version in parallel. The resulting values are concatenated and projected "
            "again. This lets the model jointly attend to information from different "
            "representation subspaces at different positions.",
        "claims": [
            {"text": "Queries keys and values are linearly projected multiple times", "importance": "must", "source": "audio"},
            {"text": "Attention is performed on projected queries keys and values in parallel", "importance": "must", "source": "audio"},
            {"text": "The outputs are concatenated and projected again", "importance": "must", "source": "audio"},
            {"text": "Multi-head attention jointly attends to different representation subspaces at different positions", "importance": "should", "source": "audio"},
            {"text": "The paper uses eight parallel attention heads", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": False,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Multi-head attention projections and parallel heads", "start_sec": 607.42, "end_sec": 665.90, "timestamp": "[10:07 - 11:05]"},
        ],
        "context_keywords": ["queries", "keys", "values", "linear projections", "parallel", "concatenated", "heads"],
        "answer_keywords": ["queries", "keys", "values", "parallel", "concatenated", "projected"],
    },
    {
        "question": "What is self-attention, and how is attention applied inside the Transformer?",
        "expected_answer":
            "Self-attention, also called intra-attention, relates positions within one "
            "sequence to compute a representation of that sequence. The Transformer "
            "uses attention in three ways: encoder-decoder attention lets decoder "
            "positions attend over encoder outputs; encoder self-attention lets each "
            "encoder position attend to encoder positions; and masked decoder "
            "self-attention prevents access to future output positions.",
        "claims": [
            {"text": "Self-attention relates different positions within a single sequence", "importance": "must", "source": "audio"},
            {"text": "Encoder-decoder attention uses queries from the decoder and keys and values from the encoder output", "importance": "must", "source": "audio"},
            {"text": "Encoder self-attention lets each encoder position attend to encoder positions", "importance": "must", "source": "audio"},
            {"text": "Decoder self-attention is masked to preserve autoregressive generation", "importance": "must", "source": "audio"},
            {"text": "The Transformer uses multi-head attention in three different ways", "importance": "should", "source": "audio"},
        ],
        "requires_continuity": True,
        "requires_timestamp": True,
        "timestamp_targets": [
            {"label": "Self-attention definition", "start_sec": 264.52, "end_sec": 275.90, "timestamp": "[04:24 - 04:35]"},
            {"label": "Applications of attention", "start_sec": 666.46, "end_sec": 746.64, "timestamp": "[11:06 - 12:26]"},
        ],
        "context_keywords": ["self-attention", "intra-attention", "encoder-decoder attention", "queries", "keys", "values", "masked"],
        "answer_keywords": ["self-attention", "encoder-decoder attention", "encoder", "decoder", "masked"],
    },
]

TEST_SET_AUDIOS = TEST_SET

print(f"Evaluation test set: {len(TEST_SET)} shared Transformer questions.")
print(f"  Timestamp questions: {sum(1 for q in TEST_SET if q.get('requires_timestamp', True))}")
print(f"  Must claims        : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'must')}")
print(f"  Should claims      : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'should')}")


Evaluation test set: 7 shared Transformer questions.
  Timestamp questions: 7
  Must claims        : 22
  Should claims      : 12


### 6.2 Retrieval Views and Audio RAG Approach Registry


In [41]:
import math
import re
from collections import defaultdict
from html import escape
from typing import Any, Callable, Iterable

try:
    import pandas as pd
except Exception as e:
    pd = None
    print(f"pandas unavailable; tables will be shown as raw lists. Error: {e}")

try:
    from IPython.display import HTML, clear_output, display
except Exception:
    HTML = None
    clear_output = None

from bert_score import score as bert_score_score

BERTSCORE_MODEL_TYPE = os.getenv("BERTSCORE_MODEL_TYPE", "distilbert-base-uncased")
CLAIM_MATCH_THRESHOLD = float(os.getenv("CLAIM_MATCH_THRESHOLD", "0.45"))
TIMESTAMP_COVERAGE_MIN_OVERLAP = float(os.getenv("TIMESTAMP_COVERAGE_MIN_OVERLAP", "0.30"))
TIMESTAMP_EVIDENCE_MIN_SCORE = float(os.getenv("TIMESTAMP_EVIDENCE_MIN_SCORE", "0.35"))
RETRIEVAL_STAGE_OPTIONS = {
    "dense": "Dense retrieval only",
    "bm25": "BM25 only",
    "hybrid": "Dense + BM25",
    "rerank": "Cross-encoder rerank",
}
APPROACH_RETRIEVAL_STAGES = {
    "whisper": ["dense", "bm25", "hybrid", "rerank"],
    "clap": ["dense", "rerank"],
}
APPROACH_STAGE_LABELS = {
    "whisper": {
        "dense": "Whisper dense retrieval only",
        "bm25": "Whisper BM25 only",
        "hybrid": "Whisper dense + BM25",
        "rerank": "Whisper dense + BM25 + cross-encoder rerank",
    },
    "clap": {
        "dense": "CLAP audio-text dense retrieval only",
        "rerank": "CLAP dense + transcript cross-encoder rerank",
    },
}


def retrieval_stage_label(approach_key: str, stage: str) -> str:
    return APPROACH_STAGE_LABELS.get(approach_key, {}).get(
        stage,
        RETRIEVAL_STAGE_OPTIONS.get(stage, stage),
    )

_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "by", "for", "from",
    "has", "have", "in", "into", "is", "it", "its", "of", "on", "or", "over",
    "that", "the", "their", "then", "there", "this", "to", "used", "using", "was",
    "were", "with", "where", "which", "while",
}


def _normalize_text(text: str) -> str:
    text = str(text or "").lower()
    text = text.replace("^", " ").replace("_", "_")
    text = re.sub(r"[^a-z0-9_\.]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _tokens(text: str) -> list[str]:
    return [t for t in _normalize_text(text).split() if len(t) > 1 and t not in _STOPWORDS]


def answer_precision_recall(candidate: str, reference: str) -> dict[str, float]:
    cand = set(_tokens(candidate))
    ref = set(_tokens(reference))
    if not cand and not ref:
        return {"precision": 1.0, "recall": 1.0, "answer_f1": 1.0}
    if not cand or not ref:
        return {"precision": 0.0, "recall": 0.0, "answer_f1": 0.0}
    overlap = len(cand & ref)
    precision = overlap / len(cand)
    recall = overlap / len(ref)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "answer_f1": f1}


def claim_match_score(claim_text: str, target_text: str) -> float:
    claim_tokens = set(_tokens(claim_text))
    target_tokens = set(_tokens(target_text))
    if not claim_tokens:
        return 0.0
    if _normalize_text(claim_text) and _normalize_text(claim_text) in _normalize_text(target_text):
        return 1.0
    token_recall = len(claim_tokens & target_tokens) / len(claim_tokens)
    token_precision = len(claim_tokens & target_tokens) / len(target_tokens) if target_tokens else 0.0
    token_f1 = 2 * token_precision * token_recall / (token_precision + token_recall) if token_precision + token_recall else 0.0
    return max(token_recall, token_f1)


def _filtered_claims(item: dict, importance: str | None = None) -> list[dict]:
    claims = item.get("claims", [])
    if importance is not None:
        claims = [c for c in claims if c.get("importance") == importance]
    return claims


def claim_recall(item: dict, target_text: str, importance: str | None = None) -> tuple[float, int, int]:
    claims = _filtered_claims(item, importance=importance)
    if not claims:
        return (math.nan, 0, 0)
    covered = sum(1 for c in claims if claim_match_score(c["text"], target_text) >= CLAIM_MATCH_THRESHOLD)
    return covered / len(claims), covered, len(claims)


def retrieved_context_text(docs: list[Any]) -> str:
    parts = []
    for i, doc in enumerate(docs, start=1):
        ts = getattr(doc, "timestamp_label", "")
        text = getattr(doc, "content", "")
        start = getattr(doc, "start_sec", None)
        end = getattr(doc, "end_sec", None)
        interval = f" [{start:.1f}-{end:.1f}s]" if start is not None and end is not None else ""
        parts.append(f"[retrieved {i}{interval}] {ts}\n{text}")
    return "\n\n".join(parts)


def _dedupe_key(doc: Any) -> tuple:
    return (
        round(float(getattr(doc, "start_sec", 0.0)), 2),
        round(float(getattr(doc, "end_sec", 0.0)), 2),
        hash(getattr(doc, "content", "")),
    )


def _append_candidate(candidates: list[tuple[str, Any]], seen: set, rerank_text: str, doc: Any) -> None:
    key = _dedupe_key(doc)
    if key not in seen:
        seen.add(key)
        candidates.append((rerank_text or getattr(doc, "content", "")[:500], doc))


def _interval(doc: Any) -> tuple[float, float] | None:
    start = getattr(doc, "start_sec", None)
    end = getattr(doc, "end_sec", None)
    if start is None or end is None:
        return None
    start = float(start)
    end = float(end)
    if end <= start:
        return None
    return (start, end)


def _overlap_seconds(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))


def _merge_intervals(intervals: list[tuple[float, float]]) -> list[tuple[float, float]]:
    if not intervals:
        return []
    ordered = sorted(intervals)
    merged = [ordered[0]]
    for start, end in ordered[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


def _target_covered(target: tuple[float, float], retrieved_intervals: list[tuple[float, float]]) -> bool:
    duration = max(1e-9, target[1] - target[0])
    covered = sum(_overlap_seconds(target, interval) for interval in _merge_intervals(retrieved_intervals))
    return (covered / duration) >= TIMESTAMP_COVERAGE_MIN_OVERLAP


def infer_timestamp_targets(item: dict, top_k_per_claim: int = 1) -> list[dict[str, Any]]:
    """Infer gold-ish evidence windows by matching each claim against Whisper chunks."""
    if item.get("timestamp_targets"):
        return item["timestamp_targets"]

    targets = []
    for claim in item.get("claims", []):
        scored = [
            (claim_match_score(claim["text"], chunk.content), chunk)
            for chunk in whisper_chunks
            if getattr(chunk, "content", "").strip()
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        for score, chunk in scored[:top_k_per_claim]:
            if score >= TIMESTAMP_EVIDENCE_MIN_SCORE:
                targets.append({
                    "claim": claim["text"],
                    "start_sec": float(chunk.start_sec),
                    "end_sec": float(chunk.end_sec),
                    "score": float(score),
                    "timestamp": chunk.timestamp_label,
                })
    return targets


def timestamp_coverage(item: dict, retrieved_docs: list[Any]) -> tuple[float, int, int, str]:
    targets = infer_timestamp_targets(item)
    if not targets:
        return (math.nan, 0, 0, "")
    retrieved_intervals = [interval for interval in (_interval(doc) for doc in retrieved_docs) if interval is not None]
    covered = 0
    labels = []
    for target in targets:
        target_interval = (float(target["start_sec"]), float(target["end_sec"]))
        hit = _target_covered(target_interval, retrieved_intervals)
        covered += int(hit)
        labels.append(f"{target.get('timestamp', f'{target_interval[0]:.1f}-{target_interval[1]:.1f}s')}:{'hit' if hit else 'miss'}")
    return covered / len(targets), covered, len(targets), "; ".join(labels)


def retrieve_whisper_by_stage(query: str, stage: str = "rerank", k: int = RETRIEVER_K, top_n: int = RERANKER_TOP_N) -> list[AudioChunk]:
    stage = stage if stage in RETRIEVAL_STAGE_OPTIONS else "rerank"
    seen: set = set()
    candidates: list[tuple[str, AudioChunk]] = []

    if stage in ("dense", "hybrid", "rerank"):
        try:
            for lc in whisper_vector_store.similarity_search(query, k=k):
                uid = lc.metadata.get("doc_id")
                chunk = whisper_docstore.get(uid) if uid else None
                if chunk is None:
                    chunk = AudioChunk(
                        content=lc.page_content,
                        start_sec=lc.metadata.get("start_sec", 0),
                        end_sec=lc.metadata.get("end_sec", 0),
                        source_file=lc.metadata.get("source", AUDIO_PATH),
                    )
                _append_candidate(candidates, seen, lc.page_content, chunk)
        except Exception as e:
            print(f"[retrieve_whisper_by_stage] Dense retrieval error: {e}")

    if stage in ("bm25", "hybrid", "rerank"):
        for score, chunk in whisper_bm25.retrieve(query, top_k=k):
            _append_candidate(candidates, seen, chunk.content[:500], chunk)

    if not candidates:
        return []
    if stage == "rerank" and RERANKER_AVAILABLE and reranker is not None:
        pairs = [(query, text) for text, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [d for _, d in candidates]), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:top_n]]
    return [d for _, d in candidates[:top_n]]


def retrieve_clap_by_stage(query: str, stage: str = "rerank", k: int = RETRIEVER_K, top_n: int = RERANKER_TOP_N) -> list[AudioSegment]:
    """
    Parameterized CLAP retrieval for evaluation.

    Valid CLAP views:
      - dense: text query in CLAP shared space retrieves audio segments
      - rerank: same CLAP dense candidates, then transcript cross-encoder reranking

    BM25 is intentionally not exposed here: it operates on transcripts, not on
    CLAP's audio-text embedding space, so it would be a different retrieval
    baseline rather than a CLAP retrieval mode.
    """
    if stage not in APPROACH_RETRIEVAL_STAGES["clap"]:
        print(f"[retrieve_clap_by_stage] Retrieval stage {stage!r} is not applicable to CLAP.")
        return []
    if not CLAP_AVAILABLE:
        print("[retrieve_clap_by_stage] CLAP not available.")
        return []

    seen: set = set()
    candidates: list[tuple[str, AudioSegment]] = []

    try:
        query_vec = encode_query_clap(query)
        result = client.query_points(
            collection_name=QDRANT_CLAP_COLLECTION,
            query=query_vec.tolist(),
            limit=k,
            with_payload=True,
        )
        for hit in result.points:
            seg = clap_docstore.get(str(hit.id))
            if seg is not None:
                _append_candidate(candidates, seen, seg.content[:500], seg)
    except Exception as e:
        print(f"[retrieve_clap_by_stage] CLAP dense retrieval error: {e}")

    if not candidates:
        return []
    if stage == "rerank" and RERANKER_AVAILABLE and reranker is not None:
        pairs = [(query, text) for text, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [d for _, d in candidates]), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:top_n]]
    return [d for _, d in candidates[:top_n]]


RAG_APPROACHES = {
    "whisper": {
        "label": "Whisper transcript RAG",
        "retriever": retrieve_whisper_by_stage,
        "description": "Dense/BM25/rerank over Whisper transcript chunks.",
    },
    "clap": {
        "label": "CLAP audio-text RAG",
        "retriever": retrieve_clap_by_stage,
        "description": "CLAP audio-text dense retrieval, optionally reranked with a text cross-encoder over attached transcripts.",
    },
}

print("Audio evaluation harness ready.")
print("Retrieval views by approach:")
for approach_key, stages in APPROACH_RETRIEVAL_STAGES.items():
    labels = ", ".join(retrieval_stage_label(approach_key, stage) for stage in stages)
    print(f"  {RAG_APPROACHES[approach_key]['label']}: {labels}")


Audio evaluation harness ready.
Retrieval views by approach:
  Whisper transcript RAG: Whisper dense retrieval only, Whisper BM25 only, Whisper dense + BM25, Whisper dense + BM25 + cross-encoder rerank
  CLAP audio-text RAG: CLAP audio-text dense retrieval only, CLAP dense + transcript cross-encoder rerank


### 6.3 Metrics


In [42]:
def compute_bertscore_batch(candidates: list[str], references: list[str]) -> list[dict[str, float]]:
    """Compute BERTScore with the installed `bert_score` package."""
    if not candidates:
        return []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    p, r, f1 = bert_score_score(
        candidates,
        references,
        lang="en",
        model_type=BERTSCORE_MODEL_TYPE,
        verbose=False,
        rescale_with_baseline=False,
        device=device,
    )
    return [
        {
            "bertscore_precision": float(pp),
            "bertscore_recall": float(rr),
            "bertscore_f1": float(ff),
            "bertscore_backend": "bert_score",
        }
        for pp, rr, ff in zip(p, r, f1)
    ]


def evaluate_single_result(item: dict, answer: str, retrieved_docs: list[Any]) -> dict[str, Any]:
    context_text = retrieved_context_text(retrieved_docs)
    lexical = answer_precision_recall(answer, item["expected_answer"])

    must_recall, must_hit, must_total = claim_recall(item, answer, importance="must")
    should_recall, should_hit, should_total = claim_recall(item, answer, importance="should")
    answer_claim_recall, answer_claim_hit, answer_claim_total = claim_recall(item, answer)
    context_recall, context_hit, context_total = claim_recall(item, context_text)
    ts_coverage, ts_hit, ts_total, ts_detail = timestamp_coverage(item, retrieved_docs)

    return {
        **lexical,
        "context_recall": context_recall,
        "timestamp_coverage": ts_coverage,
        "must_recall": must_recall,
        "should_recall": should_recall,
        "answer_claim_recall": answer_claim_recall,
        "must_covered": f"{must_hit}/{must_total}",
        "should_covered": f"{should_hit}/{should_total}",
        "context_claims_covered": f"{context_hit}/{context_total}",
        "timestamp_targets_covered": f"{ts_hit}/{ts_total}",
        "timestamp_detail": ts_detail,
        "n_retrieved": len(retrieved_docs),
        "retrieved_seconds": sum(max(0.0, float(getattr(d, "end_sec", 0)) - float(getattr(d, "start_sec", 0))) for d in retrieved_docs),
    }


def _build_eval_row(
    item: dict,
    q_idx: int,
    approach_key: str,
    stage: str,
    compute_bert_later: bool = True,
) -> tuple[dict[str, Any], str | None, str | None]:
    spec = RAG_APPROACHES[approach_key]
    stage_label = retrieval_stage_label(approach_key, stage)
    row = {
        "approach_key": approach_key,
        "approach": spec["label"],
        "retrieval_view": stage_label,
        "retrieval_stage": stage,
        "question_id": q_idx,
        "question": item["question"],
        "requires_timestamp": item.get("requires_timestamp", True),
    }
    try:
        retrieved = spec["retriever"](item["question"], stage=stage)
        answer = generate_answer(retrieved, item["question"])
        row.update(evaluate_single_result(item, answer, retrieved))
        row["answer"] = answer
        row["source_preview"] = retrieved_context_text(retrieved)[:1000]
        if compute_bert_later:
            return row, answer, item["expected_answer"]
    except Exception as e:
        row.update({
            "error": repr(e),
            "precision": math.nan,
            "recall": math.nan,
            "answer_f1": math.nan,
            "context_recall": math.nan,
            "timestamp_coverage": math.nan,
            "must_recall": math.nan,
            "should_recall": math.nan,
            "answer_claim_recall": math.nan,
            "n_retrieved": 0,
            "retrieved_seconds": 0.0,
            "answer": "",
            "source_preview": "",
            "timestamp_targets_covered": "0/0",
            "timestamp_detail": "",
        })
    return row, None, None


def _stages_for_approach(
    approach_key: str,
    stages: Iterable[str] | None = None,
    stages_by_approach: dict[str, Iterable[str]] | None = None,
) -> list[str]:
    valid = APPROACH_RETRIEVAL_STAGES.get(approach_key, list(RETRIEVAL_STAGE_OPTIONS))
    requested = list(stages_by_approach.get(approach_key, valid)) if stages_by_approach else list(stages or valid)
    return [stage for stage in requested if stage in valid]


def generate_evaluation_results(
    test_set: list[dict] = TEST_SET,
    approaches: Iterable[str] | None = None,
    stages: Iterable[str] | None = None,
    stages_by_approach: dict[str, Iterable[str]] | None = None,
    limit: int | None = None,
    compute_bert: bool = True,
) -> Any:
    approaches = list(approaches or RAG_APPROACHES.keys())
    selected_questions = test_set[:limit] if limit else test_set
    approach_stage_pairs = [
        (approach_key, stage)
        for approach_key in approaches
        for stage in _stages_for_approach(approach_key, stages, stages_by_approach)
    ]

    rows: list[dict[str, Any]] = []
    bert_candidates: list[str] = []
    bert_references: list[str] = []
    bert_row_indexes: list[int] = []

    total = len(approach_stage_pairs) * len(selected_questions)
    done = 0
    for approach_key, stage in approach_stage_pairs:
        for q_idx, item in enumerate(selected_questions, start=1):
            done += 1
            print(
                f"[{done}/{total}] {RAG_APPROACHES[approach_key]['label']} | "
                f"{retrieval_stage_label(approach_key, stage)} | Q{q_idx}"
            )
            row, bert_candidate, bert_reference = _build_eval_row(
                item=item,
                q_idx=q_idx,
                approach_key=approach_key,
                stage=stage,
                compute_bert_later=compute_bert,
            )
            if compute_bert and bert_candidate is not None and bert_reference is not None:
                bert_row_indexes.append(len(rows))
                bert_candidates.append(bert_candidate)
                bert_references.append(bert_reference)
            rows.append(row)

    if compute_bert and bert_candidates:
        bert_rows = compute_bertscore_batch(bert_candidates, bert_references)
        for idx, scores in zip(bert_row_indexes, bert_rows):
            rows[idx].update(scores)

    for row in rows:
        row.setdefault("bertscore_precision", math.nan)
        row.setdefault("bertscore_recall", math.nan)
        row.setdefault("bertscore_f1", math.nan)
        row.setdefault("bertscore_backend", "disabled" if not compute_bert else "failed")

    return pd.DataFrame(rows) if pd is not None else rows


def summarize_evaluation_results(detail_df):
    if pd is None:
        return detail_df
    if detail_df is None or len(detail_df) == 0:
        return pd.DataFrame()
    metric_cols = [
        "bertscore_f1", "precision", "recall", "answer_f1",
        "context_recall", "timestamp_coverage", "must_recall", "should_recall",
        "answer_claim_recall", "n_retrieved", "retrieved_seconds",
    ]
    return (
        detail_df
        .groupby(["approach", "retrieval_view"], dropna=False)[metric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(["approach", "retrieval_view"])
    )


def evaluate_rag_approaches(
    test_set: list[dict] = TEST_SET,
    mode_by_approach: dict[str, str] | None = None,
    approaches: Iterable[str] | None = None,
    limit: int | None = None,
    compute_bert: bool = True,
) -> tuple[Any, Any]:
    approaches = list(approaches or RAG_APPROACHES.keys())
    mode_by_approach = mode_by_approach or {
        key: ("rerank" if "rerank" in APPROACH_RETRIEVAL_STAGES.get(key, []) else APPROACH_RETRIEVAL_STAGES.get(key, ["dense"])[0])
        for key in approaches
    }
    selected_questions = test_set[:limit] if limit else test_set

    rows: list[dict[str, Any]] = []
    bert_candidates: list[str] = []
    bert_references: list[str] = []
    bert_row_indexes: list[int] = []
    for approach_key in approaches:
        stage = mode_by_approach.get(approach_key, "rerank")
        if stage not in APPROACH_RETRIEVAL_STAGES.get(approach_key, []):
            continue
        for q_idx, item in enumerate(selected_questions, start=1):
            row, bert_candidate, bert_reference = _build_eval_row(
                item=item,
                q_idx=q_idx,
                approach_key=approach_key,
                stage=stage,
                compute_bert_later=compute_bert,
            )
            if compute_bert and bert_candidate is not None and bert_reference is not None:
                bert_row_indexes.append(len(rows))
                bert_candidates.append(bert_candidate)
                bert_references.append(bert_reference)
            rows.append(row)

    if compute_bert and bert_candidates:
        bert_rows = compute_bertscore_batch(bert_candidates, bert_references)
        for idx, scores in zip(bert_row_indexes, bert_rows):
            rows[idx].update(scores)
    for row in rows:
        row.setdefault("bertscore_precision", math.nan)
        row.setdefault("bertscore_recall", math.nan)
        row.setdefault("bertscore_f1", math.nan)
        row.setdefault("bertscore_backend", "disabled" if not compute_bert else "failed")

    if pd is None:
        return rows, rows
    detail_df = pd.DataFrame(rows)
    return summarize_evaluation_results(detail_df), detail_df


def display_evaluation_tables(summary_df, detail_df, show_answers: bool = False, show_bert: bool = True):
    if pd is None:
        display(summary_df)
        return
    metric_format = {
        "bertscore_precision": "{:.3f}",
        "bertscore_recall": "{:.3f}",
        "bertscore_f1": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "answer_f1": "{:.3f}",
        "context_recall": "{:.3f}",
        "timestamp_coverage": "{:.3f}",
        "must_recall": "{:.3f}",
        "should_recall": "{:.3f}",
        "answer_claim_recall": "{:.3f}",
        "n_retrieved": "{:.1f}",
        "retrieved_seconds": "{:.1f}",
    }
    summary_cols = ["approach", "retrieval_view"]
    detail_cols = ["question_id", "approach", "retrieval_view", "requires_timestamp"]
    if show_bert:
        summary_cols += ["bertscore_f1"]
        detail_cols += ["bertscore_f1", "bertscore_backend"]
    summary_cols += [
        "precision", "recall", "context_recall", "timestamp_coverage",
        "must_recall", "should_recall", "n_retrieved", "retrieved_seconds",
    ]
    detail_cols += [
        "precision", "recall", "context_recall", "timestamp_coverage",
        "must_covered", "should_covered", "context_claims_covered",
        "timestamp_targets_covered", "question",
    ]
    if show_answers:
        detail_cols += ["answer", "timestamp_detail"]
    summary_cols = [c for c in summary_cols if c in summary_df.columns]
    detail_cols = [c for c in detail_cols if c in detail_df.columns]
    display(summary_df[summary_cols].style.format(metric_format, na_rep="n/a"))
    display(detail_df[detail_cols].style.format(metric_format, na_rep="n/a"))

print("Metric functions ready: BERTScore via bert_score, Precision, Recall, Context Recall, Timestamp Coverage, Must/Should Recall.")


Metric functions ready: BERTScore via bert_score, Precision, Recall, Context Recall, Timestamp Coverage, Must/Should Recall.


### 6.4 Generate Evaluation Answers

Run this cell when you want to refresh the evaluation cache. It performs retrieval and answer generation once, then computes metrics. The interactive table in the next cell only filters this cached dataframe, so changing dropdowns does not call the generator again.


In [43]:
# Configure this cache build before running the cell.
# Use AUDIO_EVAL_QUESTION_LIMIT in setup.env for a faster smoke test.
EVAL_APPROACHES_TO_RUN = ["whisper", "clap"]
EVAL_STAGES_BY_APPROACH = {
    "whisper": ["dense", "bm25", "hybrid", "rerank"],
    "clap": ["dense", "rerank"],
}
EVAL_QUESTION_LIMIT = min(
    int(os.getenv("AUDIO_EVAL_QUESTION_LIMIT", str(len(TEST_SET)))),
    len(TEST_SET),
)
EVAL_COMPUTE_BERTSCORE = True

EVAL_DETAIL_DF = generate_evaluation_results(
    approaches=EVAL_APPROACHES_TO_RUN,
    stages_by_approach=EVAL_STAGES_BY_APPROACH,
    limit=EVAL_QUESTION_LIMIT,
    compute_bert=EVAL_COMPUTE_BERTSCORE,
)
EVAL_SUMMARY_DF = summarize_evaluation_results(EVAL_DETAIL_DF)

print(
    f"Cached {len(EVAL_DETAIL_DF)} evaluation rows "
    f"({sum(len(EVAL_STAGES_BY_APPROACH.get(k, [])) for k in EVAL_APPROACHES_TO_RUN)} valid approach/retrieval views x {EVAL_QUESTION_LIMIT} questions)."
)
display_evaluation_tables(EVAL_SUMMARY_DF, EVAL_DETAIL_DF, show_answers=False, show_bert=True)


[1/42] Whisper transcript RAG | Whisper dense retrieval only | Q1
[2/42] Whisper transcript RAG | Whisper dense retrieval only | Q2
[3/42] Whisper transcript RAG | Whisper dense retrieval only | Q3
[4/42] Whisper transcript RAG | Whisper dense retrieval only | Q4
[5/42] Whisper transcript RAG | Whisper dense retrieval only | Q5
[6/42] Whisper transcript RAG | Whisper dense retrieval only | Q6
[7/42] Whisper transcript RAG | Whisper dense retrieval only | Q7
[8/42] Whisper transcript RAG | Whisper BM25 only | Q1
[9/42] Whisper transcript RAG | Whisper BM25 only | Q2
[10/42] Whisper transcript RAG | Whisper BM25 only | Q3
[11/42] Whisper transcript RAG | Whisper BM25 only | Q4
[12/42] Whisper transcript RAG | Whisper BM25 only | Q5
[13/42] Whisper transcript RAG | Whisper BM25 only | Q6
[14/42] Whisper transcript RAG | Whisper BM25 only | Q7
[15/42] Whisper transcript RAG | Whisper dense + BM25 | Q1
[16/42] Whisper transcript RAG | Whisper dense + BM25 | Q2
[17/42] Whisper transcript RAG

,approach,retrieval_view,bertscore_f1,precision,recall,context_recall,timestamp_coverage,must_recall,should_recall,n_retrieved,retrieved_seconds
0,CLAP audio-text RAG,CLAP audio-text dense retrieval only,0.804,0.217,0.200,0.386,0.000,0.179,0.286,5.0,50.0
1,CLAP audio-text RAG,CLAP dense + transcript cross-encoder rerank,0.802,0.232,0.199,0.195,0.048,0.179,0.214,5.0,50.0
2,Whisper transcript RAG,Whisper BM25 only,0.856,0.353,0.479,0.971,0.881,0.607,0.571,5.0,209.0
3,Whisper transcript RAG,Whisper dense + BM25,0.855,0.365,0.518,0.886,0.571,0.714,0.643,5.0,210.8
4,Whisper transcript RAG,Whisper dense + BM25 + cross-encoder rerank,0.838,0.354,0.534,0.914,0.619,0.845,0.571,5.0,210.2
5,Whisper transcript RAG,Whisper dense retrieval only,0.855,0.365,0.518,0.886,0.571,0.714,0.643,5.0,210.8


,question_id,approach,retrieval_view,requires_timestamp,bertscore_f1,bertscore_backend,precision,recall,context_recall,timestamp_coverage,must_covered,should_covered,context_claims_covered,timestamp_targets_covered,question
0,1,Whisper transcript RAG,Whisper dense retrieval only,True,0.871,bert_score,0.388,0.500,1.000,0.500,3/4,2/2,6/6,1/2,Describe the overall architecture of the Transformer model
1,2,Whisper transcript RAG,Whisper dense retrieval only,True,0.866,bert_score,0.727,0.421,1.000,1.000,1/2,0/1,3/3,1/1,What BLEU score did the Transformer achieve on WMT 2014 English-to-German translation?
2,3,Whisper transcript RAG,Whisper dense retrieval only,True,0.860,bert_score,0.333,0.484,0.600,0.000,2/3,2/2,3/5,0/3,Why does the Transformer use self-attention instead of recurrent or convolutional layers?
3,4,Whisper transcript RAG,Whisper dense retrieval only,True,0.877,bert_score,0.284,0.724,1.000,1.000,3/3,1/2,5/5,1/1,What is positional encoding and why is it necessary in the Transformer?
4,5,Whisper transcript RAG,Whisper dense retrieval only,True,0.811,bert_score,0.186,0.276,0.800,0.000,1/3,0/2,4/5,0/1,How does the decoder differ from the encoder in the Transformer?
5,6,Whisper transcript RAG,Whisper dense retrieval only,True,0.874,bert_score,0.333,0.586,1.000,1.000,3/3,2/2,5/5,1/1,Explain how multi-head attention works
6,7,Whisper transcript RAG,Whisper dense retrieval only,True,0.829,bert_score,0.302,0.633,0.800,0.500,3/4,1/1,4/5,1/2,"What is self-attention, and how is attention applied inside the Transformer?"
7,1,Whisper transcript RAG,Whisper BM25 only,True,0.822,bert_score,0.233,0.553,1.000,0.500,4/4,2/2,6/6,1/2,Describe the overall architecture of the Transformer model
8,2,Whisper transcript RAG,Whisper BM25 only,True,0.878,bert_score,0.727,0.421,1.000,1.000,1/2,0/1,3/3,1/1,What BLEU score did the Transformer achieve on WMT 2014 English-to-German translation?
9,3,Whisper transcript RAG,Whisper BM25 only,True,0.806,bert_score,0.250,0.226,1.000,0.667,1/3,0/2,5/5,2/3,Why does the Transformer use self-attention instead of recurrent or convolutional layers?


### 6.5 Interactive Check Table

This widget reads `EVAL_DETAIL_DF` from the previous cell. It does not run retrieval, reranking, generation, or BERTScore again.


In [44]:
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as e:
    widgets = None
    WIDGETS_AVAILABLE = False
    print(f"ipywidgets unavailable: {e}")


def _cached_stage_options(detail_df, approach_key: str):
    stages = list(detail_df.loc[detail_df["approach_key"] == approach_key, "retrieval_stage"].dropna().unique())
    ordered = [stage for stage in RETRIEVAL_STAGE_OPTIONS if stage in stages]
    ordered += [stage for stage in stages if stage not in ordered]
    return [(retrieval_stage_label(approach_key, stage), stage) for stage in ordered]


def _filter_cached_evaluation(detail_df, selected: list[str], mode_by_approach: dict[str, str], question_limit: int):
    if pd is None:
        return detail_df, detail_df
    if detail_df is None or len(detail_df) == 0 or not selected:
        return pd.DataFrame(), pd.DataFrame()
    mask = pd.Series(False, index=detail_df.index)
    for key in selected:
        mask |= (
            (detail_df["approach_key"] == key)
            & (detail_df["retrieval_stage"] == mode_by_approach.get(key))
        )
    filtered = detail_df.loc[mask & (detail_df["question_id"] <= question_limit)].copy()
    return summarize_evaluation_results(filtered), filtered


def _close_previous_audio_dashboard():
    """Detach callbacks and close widgets from an earlier dashboard execution."""
    previous = globals().get("_AUDIO_EVAL_DASHBOARD")
    if previous:
        for widget in previous.get("observed_widgets", []):
            widget.unobserve(previous["callback"], names="value")
        previous["button"].on_click(previous["callback"], remove=True)
        previous["output"].clear_output(wait=False)
        previous["ui"].close()
        previous["output"].close()
        return

    # Compatibility cleanup for a dashboard created by the older cell version.
    previous_ui = globals().get("evaluation_dashboard")
    if previous_ui is not None:
        try:
            previous_ui.close()
        except Exception:
            pass


def make_rag_evaluation_dashboard(detail_df=None):
    global _AUDIO_EVAL_DASHBOARD

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available. Use EVAL_DETAIL_DF directly or install ipywidgets.")
        return None
    if pd is None:
        print("pandas is not available. The cached dashboard requires pandas dataframes.")
        return None
    if detail_df is None:
        detail_df = globals().get("EVAL_DETAIL_DF")
    if detail_df is None or len(detail_df) == 0:
        print("Run the previous 'Generate Evaluation Answers' cell first to create EVAL_DETAIL_DF.")
        return None

    cached_approaches = [key for key in RAG_APPROACHES if key in set(detail_df["approach_key"])]
    approach_checks = {
        key: widgets.Checkbox(value=True, description=RAG_APPROACHES[key]["label"], indent=False)
        for key in cached_approaches
    }
    mode_dropdowns = {}
    for key in cached_approaches:
        options = _cached_stage_options(detail_df, key)
        default = "rerank" if any(value == "rerank" for _, value in options) else options[0][1]
        mode_dropdowns[key] = widgets.Dropdown(
            options=options,
            value=default,
            layout=widgets.Layout(width="330px"),
        )

    max_cached_questions = int(detail_df["question_id"].max())
    question_limit = widgets.IntSlider(
        value=max_cached_questions,
        min=1,
        max=max_cached_questions,
        step=1,
        description="Questions",
        continuous_update=False,
        layout=widgets.Layout(width="360px"),
    )
    show_bert = widgets.Checkbox(value=True, description="Show BERTScore", indent=False)
    show_answers = widgets.Checkbox(value=False, description="Show answers", indent=False)
    update_button = widgets.Button(description="Update table", button_style="primary", icon="filter")
    output = widgets.Output()
    render_state = {"running": False}

    rows = [widgets.HTML("<b>RAG approach</b>"), widgets.HTML("<b>Retrieval view</b>")]
    for key in cached_approaches:
        rows.append(approach_checks[key])
        rows.append(mode_dropdowns[key])
    selector_grid = widgets.GridBox(
        rows,
        layout=widgets.Layout(
            grid_template_columns="minmax(330px, 1fr) 350px",
            grid_gap="8px 12px",
            align_items="center",
        ),
    )

    def _render(_=None):
        if render_state["running"]:
            return
        render_state["running"] = True
        try:
            selected = [key for key, check in approach_checks.items() if check.value]
            mode_by_approach = {key: mode_dropdowns[key].value for key in selected}
            # Use the widget-native method: IPython.clear_output can append
            # output in some Jupyter frontends when called from an observer.
            output.clear_output(wait=False)
            with output:
                if not selected:
                    print("Select at least one cached RAG approach.")
                    return
                summary_df, filtered_df = _filter_cached_evaluation(
                    detail_df=detail_df,
                    selected=selected,
                    mode_by_approach=mode_by_approach,
                    question_limit=question_limit.value,
                )
                if filtered_df.empty:
                    print("No cached rows for this selection. Rerun the previous cell with those retrieval views enabled.")
                    return
                display_evaluation_tables(
                    summary_df,
                    filtered_df,
                    show_answers=show_answers.value,
                    show_bert=show_bert.value,
                )
        finally:
            render_state["running"] = False

    observed_widgets = [
        question_limit,
        show_bert,
        show_answers,
        *approach_checks.values(),
        *mode_dropdowns.values(),
    ]
    update_button.on_click(_render)
    for widget in observed_widgets:
        widget.observe(_render, names="value")

    ui = widgets.VBox([
        widgets.HTML("<h3>Interactive Audio RAG Evaluation Table</h3>"),
        widgets.HTML("<i>Cached view: changing filters does not regenerate answers.</i>"),
        selector_grid,
        widgets.HBox([question_limit, show_bert, show_answers, update_button]),
        output,
    ])
    _AUDIO_EVAL_DASHBOARD = {
        "callback": _render,
        "observed_widgets": observed_widgets,
        "button": update_button,
        "output": output,
        "ui": ui,
    }

    display(ui)
    _render()
    return ui


_close_previous_audio_dashboard()
evaluation_dashboard = make_rag_evaluation_dashboard()


### 6.6 Final Mean Comparison Table

This widget summarizes `EVAL_DETAIL_DF` by averaging every metric over the selected evaluated questions. It compares every cached approach/retrieval-view combination directly and does not rerun retrieval, generation, or BERTScore.


In [45]:
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as e:
    widgets = None
    WIDGETS_AVAILABLE = False
    print(f"ipywidgets unavailable: {e}")


AUDIO_SUMMARY_METRICS = [
    "bertscore_f1", "precision", "recall", "answer_f1",
    "context_recall", "timestamp_coverage", "must_recall", "should_recall",
    "answer_claim_recall", "n_retrieved", "retrieved_seconds",
]
AUDIO_SUMMARY_FORMAT = {
    "bertscore_f1": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "answer_f1": "{:.3f}",
    "context_recall": "{:.3f}",
    "timestamp_coverage": "{:.3f}",
    "must_recall": "{:.3f}",
    "should_recall": "{:.3f}",
    "answer_claim_recall": "{:.3f}",
    "n_retrieved": "{:.1f}",
    "retrieved_seconds": "{:.1f}",
}


def _audio_mean_summary(detail_df, selected_keys, selected_views, question_limit, sort_metric, descending=True):
    if pd is None or detail_df is None or len(detail_df) == 0:
        return pd.DataFrame()
    df = detail_df.copy()
    if selected_keys:
        df = df[df["approach_key"].isin(selected_keys)]
    if selected_views:
        df = df[df["retrieval_view"].isin(selected_views)]
    if "question_id" in df.columns:
        df = df[df["question_id"] <= question_limit]
    if df.empty:
        return pd.DataFrame()

    metric_cols = [c for c in AUDIO_SUMMARY_METRICS if c in df.columns]
    grouped = df.groupby(["approach", "retrieval_view"], dropna=False)
    summary = grouped[metric_cols].mean(numeric_only=True).reset_index()
    counts = grouped["question_id"].nunique().reset_index(name="n_questions")
    summary = summary.merge(counts, on=["approach", "retrieval_view"], how="left")
    if sort_metric in summary.columns:
        summary = summary.sort_values(sort_metric, ascending=not descending, na_position="last")
    summary = summary.reset_index(drop=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    ordered_cols = ["rank", "approach", "retrieval_view", "n_questions"] + metric_cols
    return summary[[c for c in ordered_cols if c in summary.columns]]


def _close_previous_audio_summary_dashboard():
    previous = globals().get("_AUDIO_FINAL_SUMMARY_DASHBOARD")
    if not previous:
        return
    for widget in previous.get("observed_widgets", []):
        widget.unobserve(previous["callback"], names="value")
    previous["button"].on_click(previous["callback"], remove=True)
    previous["output"].clear_output(wait=False)
    previous["ui"].close()
    previous["output"].close()


def make_audio_mean_summary_dashboard(detail_df=None):
    global _AUDIO_FINAL_SUMMARY_DASHBOARD

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available. Showing a static mean summary instead.")
        detail_df = globals().get("EVAL_DETAIL_DF") if detail_df is None else detail_df
        display(summarize_evaluation_results(detail_df))
        return None
    if pd is None:
        print("pandas is not available. The mean summary table requires pandas.")
        return None
    if detail_df is None:
        detail_df = globals().get("EVAL_DETAIL_DF")
    if detail_df is None or len(detail_df) == 0:
        print("Run the 'Generate Evaluation Answers' cell first to create EVAL_DETAIL_DF.")
        return None

    cached_keys = [key for key in RAG_APPROACHES if key in set(detail_df["approach_key"])]
    approach_checks = {
        key: widgets.Checkbox(value=True, description=RAG_APPROACHES[key]["label"], indent=False)
        for key in cached_keys
    }
    view_options = sorted(detail_df["retrieval_view"].dropna().unique().tolist())
    view_select = widgets.SelectMultiple(
        options=view_options,
        value=tuple(view_options),
        description="Views",
        rows=min(8, max(3, len(view_options))),
        layout=widgets.Layout(width="460px"),
    )
    max_questions = int(detail_df["question_id"].max())
    question_limit = widgets.IntSlider(
        value=max_questions,
        min=1,
        max=max_questions,
        step=1,
        description="Questions",
        continuous_update=False,
        layout=widgets.Layout(width="360px"),
    )
    metric_options = [(c.replace("_", " "), c) for c in AUDIO_SUMMARY_METRICS if c in detail_df.columns]
    default_metric = "bertscore_f1" if "bertscore_f1" in detail_df.columns else metric_options[0][1]
    sort_metric = widgets.Dropdown(options=metric_options, value=default_metric, description="Sort by")
    descending = widgets.Checkbox(value=True, description="Descending", indent=False)
    show_bert = widgets.Checkbox(value=True, description="Show BERTScore", indent=False)
    update_button = widgets.Button(description="Update summary", button_style="primary", icon="bar-chart")
    output = widgets.Output()
    render_state = {"running": False}

    def _render(_=None):
        if render_state["running"]:
            return
        render_state["running"] = True
        try:
            output.clear_output(wait=False)
            with output:
                selected_keys = [key for key, check in approach_checks.items() if check.value]
                if not selected_keys:
                    print("Select at least one approach.")
                    return
                selected_views = list(view_select.value)
                summary = _audio_mean_summary(
                    detail_df,
                    selected_keys,
                    selected_views,
                    question_limit.value,
                    sort_metric.value,
                    descending.value,
                )
                if summary.empty:
                    print("No cached rows for this summary selection.")
                    return
                display_cols = list(summary.columns)
                if not show_bert.value:
                    display_cols = [c for c in display_cols if not c.startswith("bertscore")]
                display(summary[display_cols].style.format(AUDIO_SUMMARY_FORMAT, na_rep="n/a"))
        finally:
            render_state["running"] = False

    approach_box = widgets.VBox(list(approach_checks.values()))
    controls = widgets.HBox([
        approach_box,
        view_select,
        widgets.VBox([question_limit, sort_metric, widgets.HBox([descending, show_bert, update_button])]),
    ])
    observed_widgets = [
        *approach_checks.values(),
        view_select,
        question_limit,
        sort_metric,
        descending,
        show_bert,
    ]
    update_button.on_click(_render)
    for widget in observed_widgets:
        widget.observe(_render, names="value")

    ui = widgets.VBox([
        widgets.HTML("<h3>Final Mean Audio RAG Comparison Table</h3>"),
        widgets.HTML("<i>Means are computed over the selected evaluated questions. No retrieval, generation, or BERTScore is rerun.</i>"),
        controls,
        output,
    ])
    _AUDIO_FINAL_SUMMARY_DASHBOARD = {
        "callback": _render,
        "observed_widgets": observed_widgets,
        "button": update_button,
        "output": output,
        "ui": ui,
    }
    display(ui)
    _render()
    return ui


_close_previous_audio_summary_dashboard()
mean_summary_dashboard = make_audio_mean_summary_dashboard()


### 6.7 Answer Generation Showcase

Use this final cell to generate side-by-side answer examples after the audio indexes and generator are loaded. The examples reuse the same approach/retrieval controls as the evaluation table.


In [46]:
SHOWCASE_QUESTIONS = [
    "Describe the overall architecture of the Transformer model",
    "What is positional encoding and why is it necessary in the Transformer?",
    "Explain how multi-head attention works",
]

def _doc_preview(doc: Any, max_chars: int = 220) -> str:
    ts = getattr(doc, "timestamp_label", "")
    text = re.sub(r"\s+", " ", getattr(doc, "content", "")).strip()
    return f"{ts} " + text[:max_chars]


def showcase_answer_generation(
    questions: list[str] = SHOWCASE_QUESTIONS,
    approaches: Iterable[str] = ("whisper", "clap"),
    mode_by_approach: dict[str, str] | None = None,
    max_sources: int = 3,
):
    mode_by_approach = mode_by_approach or {key: "rerank" for key in approaches}
    rows = []
    for question in questions:
        for approach_key in approaches:
            spec = RAG_APPROACHES[approach_key]
            stage = mode_by_approach.get(approach_key, "rerank")
            docs = spec["retriever"](question, stage=stage)
            answer = generate_answer(docs, question)
            rows.append({
                "question": question,
                "approach": spec["label"],
                "retrieval_view": RETRIEVAL_STAGE_OPTIONS.get(stage, stage),
                "answer": answer,
                "sources": "\n".join(_doc_preview(d) for d in docs[:max_sources]),
            })

    if pd is not None:
        df = pd.DataFrame(rows)
        display(df.style.set_properties(subset=["answer", "sources"], **{"white-space": "pre-wrap"}))
    elif HTML is not None:
        html_rows = []
        for row in rows:
            html_rows.append(
                "<tr>"
                f"<td>{escape(row['question'])}</td>"
                f"<td>{escape(row['approach'])}</td>"
                f"<td>{escape(row['retrieval_view'])}</td>"
                f"<td><pre>{escape(row['answer'])}</pre></td>"
                f"<td><pre>{escape(row['sources'])}</pre></td>"
                "</tr>"
            )
        display(HTML("<table>" + "".join(html_rows) + "</table>"))
    else:
        for row in rows:
            print("=" * 100)
            print(row["question"])
            print(row["approach"], "|", row["retrieval_view"])
            print(row["answer"])
            print(row["sources"])


showcase_answer_generation(
    questions=SHOWCASE_QUESTIONS,
    mode_by_approach={
        "whisper": "rerank",
        "clap": "rerank",
    },
)


,question,approach,retrieval_view,answer,sources
0,Describe the overall architecture of the Transformer model,Whisper transcript RAG,Cross-encoder rerank,"The Transformer follows an encoder-decoder architecture. Here's its overall structure: 1. **Encoder**: Composed of a stack of six identical layers, each containing: - A multi-head self-attention mechanism. - A simple position-wise fully connected feed-forward network. Each sub-layer is surrounded by residual connections and followed by layer normalization. 2. **Decoder**: Also composed of a stack of six identical layers, similar to the encoder but with an additional multi-head attention sub-layer that attends to the encoder's output. It also includes: - A multi-head self-attention mechanism. - A position-wise fully connected feed-forward network. Each sub-layer is surrounded by residual connections and followed by layer normalization. 3. **Auto-regressive Generation**: At each step, the model consumes previously generated symbols as additional input when generating the next symbol. The Transformer uses attention mechanisms instead of recurrence or convolutions for sequence transduction tasks. It was introduced in the paper ""Attention is All You Need"".","[05:47 – 06:32] At each step, the model is auto-regressive, consuming the previously generated symbols as additional input when generating the next. The transformer follows this overall architecture using stacks' self-attention and poin [05:07 – 05:47] on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution. In the following sections, we will describe the transformer, motivate self-attention, and discuss i [18:33 – 19:16] Not only do individual attention heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic and semantic structure of the sentences. Seven conclusion. In this work, we presen"
1,Describe the overall architecture of the Transformer model,CLAP audio-text RAG,Cross-encoder rerank,"The Transformer model consists of an encoder and a decoder, each containing multiple layers. Each layer includes attention sub-layers and a fully connected feedforward network applied to each position separately and identically. This feedforward network involves two linear transformations with a real U activation in between.","[12:32 – 12:42] In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feedforward network, which is applied to each position separately and identically. This consists of two linear [09:28 – 09:38] [08:24 – 08:34]"
2,What is positional encoding and why is it necessary in the Transformer?,Whisper transcript RAG,Cross-encoder rerank,"Positional encoding is a method used in the Transformer model to provide information about the relative or absolute position of tokens in a sequence. It's necessary because the Transformer contains no recurrence (like LSTMs) or convolution, so it needs some way to understand the order of the sequence. Positional encodings are added to input embeddings at the bottoms of the encoder and decoder stacks. In this work, positional encoding is implemented using sine and cosine functions of different frequencies. The necessity arises from the fact that the model must make use of the order of the sequence to understand context and dependencies between distant positions. Without it, the model would treat all inputs as independent of their position in the sequence.","[14:29 – 15:04] In this work, we use sine and cosine functions of different frequencies. The positional encoding for position and dimension 2i is the sine of the position divided by 10,000 to the power of 2i over D model. The positional [15:04 – 15:39] We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset k, the positional encoding at position plus k can be represented as a li